# Obtain a Dataset and Frame the Predictive Problem

In [ ]:
import subprocess, sys

# Install required packages (ensures reproducibility in fresh environments)
for pkg in ['xgboost', 'lightgbm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")

## 1.1 Load the Dataset

The dataset is from the **Austin Animal Center** — the largest no-kill municipal shelter in the US. It merges intake and outcome records for animals from October 2013 onwards.

- **Source:** [Kaggle – Austin Animal Center Shelter Intakes and Outcomes](https://www.kaggle.com/datasets/aaronschlegel/austin-animal-center-shelter-intakes-and-outcomes)
- **License:** Open Database License (ODbL 1.0)

In [ ]:
df = pd.read_csv('aac_intakes_outcomes.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

In [ ]:
# Inspect columns and data types
print("Columns and dtypes:\n")
print(df.dtypes.to_string())
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Check the target variable distribution
print("Outcome type distribution:\n")
print(df['outcome_type'].value_counts())
print(f"\nUnique outcomes: {df['outcome_type'].nunique()}")

## 1.2 Frame the Predictive Problem

**Problem type:** Binary classification

**Target variable:** `outcome_type` → binarised as:
- **1 (Adopted):** `outcome_type == 'Adoption'`
- **0 (Not Adopted):** All other outcomes (Transfer, Return to Owner, Euthanasia, Died, etc.)

**Prediction goal:** Given information known **at intake time**, predict whether an animal will be adopted.

**Success metrics:**

- **Primary — F1-score:** The harmonic mean of precision and recall, capturing the trade-off between false positives and false negatives in a single metric. A false positive (predicting adoption when the animal is not adopted) could lead the shelter to under-allocate resources to an at-risk animal; a false negative (missing a likely adoption) could result in unnecessary intervention for an animal that would have been adopted anyway. F1 penalises both error types equally, making it well-suited for this imbalanced setting where accuracy alone would be misleading.

- **Secondary — AUC-ROC:** Measures the model's discrimination ability across all classification thresholds, rather than at a single operating point. A high AUC indicates the model consistently ranks adopted animals higher than non-adopted ones, regardless of the threshold chosen — making it useful for comparing models without committing to a specific decision boundary.

- **Supporting — Precision and Recall:** Examined individually to analyse Type I and Type II errors. **Recall** is particularly important in this context: a false negative means an adoptable animal is incorrectly flagged as unlikely to be adopted, potentially leading to premature euthanasia or deprioritised care — a high-cost error the shelter would want to minimise. **Precision** matters when shelter resources are limited: a false positive means the shelter invests promotional effort or kennel space expecting an adoption that does not materialise, wasting scarce capacity that could have been directed elsewhere. Inspecting both helps identify whether the model errs more towards over-predicting or under-predicting adoption.

**Constraints:**
- Only features available at **intake time** can be used as predictors (no outcome-time features — this avoids data leakage)
- Model should be interpretable enough for shelter staff to act on predictions
- Reproducibility: all experiments use a fixed random seed (`RANDOM_STATE = 42`), and the full pipeline is re-runnable from source
- Fairness: predictions should not systematically disadvantage certain breeds or animal types without justification

**Assumptions and limitations:**
- The dataset reflects one shelter (Austin Animal Center) — generalisability to other shelters is limited
- "Not Adopted" is a heterogeneous group (transfers, returns to owner, euthanasia) — grouping them may obscure different dynamics
- Historical patterns may shift over time (concept drift)

In [ ]:
# Create binary target variable
# First, check for missing outcome_type values
missing_outcome = df['outcome_type'].isnull().sum()
print(f"Missing outcome_type values: {missing_outcome} ({missing_outcome/len(df):.2%})")

# Drop rows with unknown outcomes — these cannot be labelled
if missing_outcome > 0:
    df = df.dropna(subset=['outcome_type']).reset_index(drop=True)
    print(f"Dropped {missing_outcome} rows with missing outcomes")
    print(f"Dataset after cleaning: {df.shape[0]:,} rows")

# Now safely create binary target
df['adopted'] = (df['outcome_type'] == 'Adoption').astype(int)

print(f"\nBinary target distribution:")
print(df['adopted'].value_counts())
print(f"\nAdoption rate: {df['adopted'].mean():.1%}")

## 1.3 Feature Selection Rationale (Leakage Prevention)

To avoid **data leakage**, we must only use features available at **intake time**. Any feature that is determined at or after the outcome event must be excluded.

**Target handling:** `outcome_type` is used solely to construct the binary target (`adopted`), then excluded from the feature set — it is the label, not a predictor.

| Keep (intake-time) | Drop | Reason |
|---|---|---|
| `animal_type`, `breed`, `color` | `outcome_type` | Used to create target, then excluded |
| `sex_upon_intake` | `outcome_subtype`, `outcome_number` | Outcome-time only |
| `age_upon_intake_(days)` | `sex_upon_outcome` | Outcome-time only |
| `intake_condition`, `intake_type` | `age_upon_outcome`, `age_upon_outcome_(days)`, `age_upon_outcome_(years)`, `age_upon_outcome_age_group` | Outcome-time only |
| `intake_month`, `intake_weekday`, `intake_hour` | `outcome_datetime`, `outcome_month`, `outcome_year`, `outcome_monthyear`, `outcome_weekday`, `outcome_hour` | Outcome-time only |
| `found_location` | `time_in_shelter`, `time_in_shelter_days` | Requires outcome datetime |
| | `animal_id_outcome`, `animal_id_intake` | Identifiers, not features |
| | `intake_year` | Temporal overfitting risk |
| | `intake_datetime` | Raw timestamp, cyclical components already extracted |
| | `intake_monthyear`, `intake_number` | Redundant / derived temporal fields |
| | `date_of_birth`, `dob_year`, `dob_month`, `dob_monthyear` | Redundant with `age_upon_intake_(days)` |
| | `age_upon_intake`, `age_upon_intake_(years)`, `age_upon_intake_age_group` | Redundant with `age_upon_intake_(days)` |
| | `count` | Constant column (all values = 1) |

In [ ]:
# Ensure binary target exists (safe to re-run)
df['adopted'] = (df['outcome_type'] == 'Adoption').astype(int)

# Define intake-time features to keep
intake_features = [
    'animal_type', 'breed', 'color',
    'sex_upon_intake', 'age_upon_intake_(days)',
    'intake_condition', 'intake_type', 'found_location',
    'intake_month', 'intake_weekday', 'intake_hour',
]

# Dropped columns and rationale:
# TARGET SOURCE: outcome_type used to create 'adopted', then excluded (it IS the label)
# OUTCOME-TIME (leakage): outcome_subtype, outcome_number, sex_upon_outcome,
#   age_upon_outcome (all variants), outcome_datetime and derivatives
#   (outcome_month/year/monthyear/weekday/hour), time_in_shelter, time_in_shelter_days
# IDENTIFIERS: animal_id_outcome, animal_id_intake
# TEMPORAL OVERFITTING: intake_year
# RAW TIMESTAMPS: intake_datetime (cyclical components already extracted)
# REDUNDANT TEMPORAL: intake_monthyear, intake_number
# REDUNDANT AGE: age_upon_intake (text), age_upon_intake_(years), age_upon_intake_age_group
# REDUNDANT DOB: date_of_birth, dob_year, dob_month, dob_monthyear (all captured by age)
# ZERO VARIANCE: count (constant = 1)

df_model = df[intake_features + ['adopted']].copy()
print(f"Modelling dataset shape: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")
print(f"\nFeatures retained ({len(intake_features)}):")
for f in intake_features:
    print(f"  - {f}")

# Explore the Data to Gain Insights

## 2.1 Raw Dataset Overview

In [ ]:
# Raw dataset overview — original 41 columns + 1 derived target ('adopted') = 42 columns
# Note: 'adopted' was created in Section 1 from outcome_type. It is included in df
# but will NOT be used as a feature — it is the target label.
print(f"=== Raw Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns ===")
print(f"    (41 original columns + 1 derived target 'adopted')\n")
print(df.dtypes.to_string())
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Missingness across the FULL raw dataset (42 columns = 41 original + 'adopted')
raw_missing = df.isnull().sum()
raw_missing_pct = (raw_missing / len(df) * 100).round(2)
raw_missing_df = pd.DataFrame({'Missing Count': raw_missing, 'Missing %': raw_missing_pct})
raw_missing_df = raw_missing_df.sort_values('Missing %', ascending=False)

print("=== Missingness Across All Columns (41 original + 1 derived target) ===\n")
print(raw_missing_df[raw_missing_df['Missing Count'] > 0].to_string())
print(f"\nColumns with no missing values: {(raw_missing == 0).sum()} / {len(raw_missing)}")
print(f"\nNote: outcome_subtype has >50% missing values — this confirms the variable is")
print(f"unreliable on data quality grounds alone, even before considering leakage.")

# Visualise — show ALL columns so we can compare missing vs complete
fig, ax = plt.subplots(figsize=(14, 8))
colors = ['coral' if v > 0 else '#c0c0c0' for v in raw_missing_df['Missing %'].values]
bars = ax.barh(range(len(raw_missing_df)), raw_missing_df['Missing %'].values, color=colors)
ax.set_yticks(range(len(raw_missing_df)))
ax.set_yticklabels(raw_missing_df.index, fontsize=8)
ax.set_xlabel('Missing (%)', fontsize=11)
ax.set_title('Missing Values Across All Columns (41 original + 1 derived target)', fontsize=13)
ax.invert_yaxis()

# Annotate non-zero bars with percentages
for i, (val, col_name) in enumerate(zip(raw_missing_df['Missing %'].values, raw_missing_df.index)):
    if val > 0:
        ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

We now examine the full distribution of outcome types before binarising into Adopted vs Not Adopted. This helps us understand the composition of the negative class and the relative prevalence of adoption compared to other outcomes such as Transfer, Return to Owner, and Euthanasia.

In [ ]:
# Full outcome type distribution (before binarisation)
outcome_counts = df['outcome_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart — with percentage labels
colors_outcome = sns.color_palette('Set2', len(outcome_counts))
bars = axes[0].bar(range(len(outcome_counts)), outcome_counts.values, color=colors_outcome)
axes[0].set_xticks(range(len(outcome_counts)))
axes[0].set_xticklabels(outcome_counts.index, rotation=45, ha='right', fontsize=10)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('All Outcome Types (Raw)', fontsize=13)
for i, v in enumerate(outcome_counts.values):
    axes[0].text(i, v + 300, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=9, fontweight='bold')

# Donut chart — cleaner than pie for many categories
# Group small categories for readability
threshold = 0.02  # group categories < 2%
major = outcome_counts[outcome_counts / len(df) >= threshold]
minor_sum = outcome_counts[outcome_counts / len(df) < threshold].sum()
if minor_sum > 0:
    plot_data = pd.concat([major, pd.Series({'Other': minor_sum})])
else:
    plot_data = major

donut_colors = sns.color_palette('Set2', len(plot_data))
wedges, texts, autotexts = axes[1].pie(
    plot_data.values, labels=plot_data.index, colors=donut_colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    textprops={'fontsize': 11}
)
# Make percentage text bold and larger
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
# Add centre circle for donut effect
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
axes[1].add_artist(centre_circle)
axes[1].set_title('Outcome Type Proportions', fontsize=13)

plt.tight_layout()
plt.show()

print("=== Full Outcome Distribution ===")
for outcome, count in outcome_counts.items():
    print(f"  {outcome}: {count:,} ({count/len(df):.1%})")

# Metric confirmation from outcome distribution
adoption_pct = outcome_counts.get('Adoption', 0) / len(df) * 100
not_adopted_pct = 100 - adoption_pct
print(f"\n=== Metric Selection Confirmed by Outcome Distribution ===")
print(f"\nAfter binarisation: Adopted ~{adoption_pct:.0f}% vs Not Adopted ~{not_adopted_pct:.0f}%")
print(f"\n1. F1-score justified: The class split (~{adoption_pct:.0f}/{not_adopted_pct:.0f}) shows moderate")
print(f"   imbalance. Accuracy would be misleading — a naive classifier predicting all as")
print(f"   'Not Adopted' would achieve ~{not_adopted_pct:.0f}% accuracy. F1 guards against this by")
print(f"   requiring both precision and recall to be high.")
print(f"\n2. AUC-ROC justified: The 'Not Adopted' class is heterogeneous — it groups")
print(f"   Transfer ({outcome_counts.get('Transfer',0)/len(df):.1%}), "
      f"Return to Owner ({outcome_counts.get('Return to Owner',0)/len(df):.1%}), "
      f"Euthanasia ({outcome_counts.get('Euthanasia',0)/len(df):.1%}), etc.")
print(f"   These very different outcomes merged into one class may blur the decision boundary.")
print(f"   AUC-ROC evaluates discrimination across all thresholds, making it robust to this")
print(f"   heterogeneity — a model can still rank adopted animals higher even if the boundary")
print(f"   between Adoption and the diverse 'Not Adopted' group is not clean at any single threshold.")

Next, we compare intake-time and outcome-time features to understand how animals change during their shelter stay, and to further validate which variables constitute leakage. We also examine the distribution of time spent in the shelter across different outcome types.

In [ ]:
# Explore intake vs outcome feature relationships in the raw data
# Do animals change neuter status between intake and outcome?
neuter_change = (df['sex_upon_intake'] != df['sex_upon_outcome']).sum()
neuter_change_pct = neuter_change / len(df) * 100
print(f"=== Intake vs Outcome Comparison ===\n")
print(f"Animals whose sex/neuter status changed between intake and outcome: "
      f"{neuter_change:,} ({neuter_change_pct:.1f}%)")
print("→ This confirms sex_upon_outcome cannot be used as a feature (it changes after intake)\n")

# Time in shelter distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shelter_days = df['time_in_shelter_days']
axes[0].hist(shelter_days[shelter_days <= 60], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Days in Shelter')
axes[0].set_ylabel('Count')
axes[0].set_title('Time in Shelter Distribution (≤ 60 days)')
axes[0].axvline(x=shelter_days.median(), color='red', linestyle='--',
                label=f'Median: {shelter_days.median():.1f} days')
axes[0].legend()

# Time in shelter by outcome
for outcome in ['Adoption', 'Transfer', 'Return to Owner', 'Euthanasia']:
    subset = df[df['outcome_type'] == outcome]['time_in_shelter_days']
    subset = subset[subset <= 60]
    axes[1].hist(subset, bins=40, alpha=0.5, label=f'{outcome} (med={subset.median():.1f}d)')
axes[1].set_xlabel('Days in Shelter')
axes[1].set_ylabel('Count')
axes[1].set_title('Time in Shelter by Outcome Type (≤ 60 days)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Median time in shelter: {shelter_days.median():.1f} days")
print(f"Mean time in shelter: {shelter_days.mean():.1f} days")
print(f"Max time in shelter: {shelter_days.max():.0f} days")
print(f"\nNote: Although time_in_shelter shows clear separation between outcome types and would")
print(f"likely be highly predictive, it cannot be used as a feature because it is only known")
print(f"after the outcome event (time_in_shelter = outcome_datetime − intake_datetime).")
print(f"Including it would constitute direct data leakage.")

In [ ]:
# Duplicate animal IDs — repeat visitors to the shelter
repeat_animals = df['animal_id_intake'].value_counts()
n_unique = df['animal_id_intake'].nunique()
n_repeat = (repeat_animals > 1).sum()

print("=== Repeat Shelter Visits ===\n")
print(f"Total rows: {len(df):,}")
print(f"Unique animals: {n_unique:,}")
print(f"Animals with multiple visits: {n_repeat:,} ({n_repeat/n_unique:.1%} of unique animals)")
print(f"Max visits by a single animal: {repeat_animals.max()}")

fig, ax = plt.subplots(figsize=(8, 4))
visit_dist = repeat_animals.value_counts().sort_index()
ax.bar(visit_dist.index[:10], visit_dist.values[:10], color='steelblue')
ax.set_xlabel('Number of Shelter Visits')
ax.set_ylabel('Number of Animals')
ax.set_title('Distribution of Repeat Shelter Visits')
for i, (visits, count) in enumerate(visit_dist.head(10).items()):
    ax.text(visits, count + 50, f'{count:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Impact on modelling assumptions
print(f"\n=== Impact on Modelling Assumptions ===")
print(f"\n1. Independence violation: Approximately {n_repeat/n_unique:.1%} of animals appear")
print(f"   multiple times in the dataset. Because multiple records correspond to the same")
print(f"   animal, observations are not strictly independent. This violates the i.i.d.")
print(f"   assumption underlying most machine learning classifiers and may inflate performance")
print(f"   if repeat records are split across training and testing sets.")
print(f"\n2. Long tail effect: The visit distribution is heavily right-skewed — most animals")
print(f"   enter the shelter only once, while a small number return up to {repeat_animals.max()} times.")
print(f"   These frequent returners are disproportionately represented in the data, which risks")
print(f"   biasing the model toward patterns specific to animals that cycle through the shelter")
print(f"   system rather than learning generalisable adoption predictors.")
print(f"\nMitigation: To address this, we considered three approaches: (a) keeping only the first")
print(f"visit per animal for strict independence, (b) grouping train/test splits by animal_id,")
print(f"or (c) accepting the violation as a limitation. We will adopt option (b) using")
print(f"GroupShuffleSplit by animal_id in Section 3 — this ensures the same animal never appears")
print(f"in both train and test sets, directly preventing inflated metrics from repeat records,")
print(f"while preserving the full dataset size and the useful variation across visits.")

## 2.2 Modelling Dataset: Missingness and Overview

In [ ]:
# Modelling dataset overview and missingness
print(f"=== Modelling Dataset: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns ===\n")

model_missing = df_model.isnull().sum()
model_missing_pct = (model_missing / len(df_model) * 100).round(2)
model_missing_df = pd.DataFrame({'Missing Count': model_missing, 'Missing %': model_missing_pct})
model_missing_df = model_missing_df[model_missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(model_missing_df) > 0:
    print(f"=== Missing Values (NaN) in Modelling Features ===\n")
    print(model_missing_df.to_string())
else:
    print("No null values in the modelling dataset.")

# Check for 'Unknown' string values acting as implicit missingness
unknown_counts = {}
for col in df_model.select_dtypes(include='object').columns:
    n_unknown = (df_model[col] == 'Unknown').sum()
    if n_unknown > 0:
        unknown_counts[col] = n_unknown
if unknown_counts:
    print(f"\n=== Implicit Missingness ('Unknown' string values) ===\n")
    for col, count in unknown_counts.items():
        print(f"  {col}: {count:,} ({count/len(df_model):.2%})")

print(f"\n=== Handling Strategy (to be applied in Section 3) ===")
print(f"\n• sex_upon_intake: Contains 'Unknown' string values and potential nulls. Both will")
print(f"  be preserved as a distinct category during feature engineering, since the fact that")
print(f"  the status is unknown may itself carry predictive signal (e.g. feral animals).")
print(f"• age_upon_intake_(days): Any missing values will be imputed using the median, as age")
print(f"  is a key predictor and median imputation is robust to the right-skewed distribution")
print(f"  observed in Section 2.5.")
print(f"• Categorical features with minimal missingness: will be imputed with the mode")
print(f"  (most frequent category) inside the preprocessing pipeline to avoid data loss.")
print(f"\nNote: All imputation will be fit on the training set only and applied to the test")
print(f"set, preventing information leakage from the test split.")

## 2.3 Target Variable Distribution (Class Imbalance)

We examine the binary target distribution to assess class imbalance, which directly informs our choice of evaluation metrics and whether resampling or class-weight adjustments are needed during modelling.

In [ ]:
# Target variable distribution
fig, ax = plt.subplots(figsize=(8, 5))

target_counts = df_model['adopted'].value_counts()
labels = ['Not Adopted (0)', 'Adopted (1)']
colors = ['#e74c3c', '#2ecc71']
ax.bar(labels, target_counts.values, color=colors)
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 300, f'{v:,}\n({v/len(df_model):.1%})', ha='center', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Target Variable Distribution')

plt.tight_layout()
plt.show()

imbalance_ratio = target_counts.min() / target_counts.max()
print(f"Imbalance ratio (minority/majority): {imbalance_ratio:.2f}")
print(f"→ {'Moderate imbalance — F1 and stratified splits recommended' if imbalance_ratio < 0.8 else 'Reasonably balanced'}")

## 2.4 Categorical Feature Distributions

We explore the cardinality and adoption rates across categorical features to identify which categories carry the strongest predictive signal. Low-cardinality features are examined individually, while high-cardinality features (breed, color) are shown for the top 15 most frequent categories.

In [ ]:
# Categorical feature cardinality
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print("=== Categorical Features ===\n")
for col in cat_cols:
    n_unique = df_model[col].nunique()
    top_val = df_model[col].value_counts().index[0]
    top_pct = df_model[col].value_counts(normalize=True).iloc[0] * 100
    print(f"  {col}: {n_unique} unique | top: '{top_val}' ({top_pct:.1f}%)")

In [ ]:
# Adoption rate by low-cardinality categorical features
low_card_cats = ['animal_type', 'sex_upon_intake', 'intake_condition', 'intake_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(low_card_cats):
    adoption_rate = df_model.groupby(col)['adopted'].mean().sort_values(ascending=False)
    counts = df_model[col].value_counts()
    
    bars = axes[i].bar(range(len(adoption_rate)), adoption_rate.values, color='steelblue')
    axes[i].set_xticks(range(len(adoption_rate)))
    axes[i].set_xticklabels(adoption_rate.index, rotation=45, ha='right', fontsize=9)
    axes[i].set_ylabel('Adoption Rate')
    axes[i].set_title(f'Adoption Rate by {col}')
    axes[i].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--', 
                     label=f'Overall: {df_model["adopted"].mean():.1%}')
    axes[i].legend(fontsize=8)
    
    # Annotate with counts
    for j, (rate, cat) in enumerate(zip(adoption_rate.values, adoption_rate.index)):
        n = counts.get(cat, 0)
        axes[i].text(j, rate + 0.01, f'n={n:,}', ha='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# High-cardinality features: breed and color (top 15)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, col in enumerate(['breed', 'color']):
    top_cats = df_model[col].value_counts().head(15).index
    subset = df_model[df_model[col].isin(top_cats)]
    adoption_rate = subset.groupby(col)['adopted'].mean().sort_values(ascending=False)
    
    axes[i].barh(range(len(adoption_rate)), adoption_rate.values, color='steelblue')
    axes[i].set_yticks(range(len(adoption_rate)))
    axes[i].set_yticklabels(adoption_rate.index, fontsize=9)
    axes[i].set_xlabel('Adoption Rate')
    axes[i].set_title(f'Adoption Rate by {col} (Top 15)')
    axes[i].axvline(x=df_model['adopted'].mean(), color='red', linestyle='--',
                     label=f'Overall: {df_model["adopted"].mean():.1%}')
    axes[i].legend(fontsize=8)
    axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Breed cardinality: {df_model['breed'].nunique()} unique values")
print(f"Color cardinality: {df_model['color'].nunique()} unique values")
print("→ High cardinality — will need grouping or encoding in Section 3")
print(f"\nNote: Color appears to have a modest effect on adoption probability — the top 15")
print(f"colors cluster relatively close to the overall adoption rate, suggesting color alone")
print(f"is not a strong discriminator. It may still contribute useful signal in combination")
print(f"with other features, but is unlikely to be a primary predictor.")

## 2.5 Numeric Feature Distribution and Outliers

We analyse the distribution of `age_upon_intake_(days)` — the only continuous numeric feature — to assess its shape, identify outliers, and examine how age distributions differ between adopted and non-adopted animals.

In [ ]:
# Age distribution by adoption outcome
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
for label, color in zip([0, 1], ['#e74c3c', '#2ecc71']):
    subset = df_model[df_model['adopted'] == label]['age_upon_intake_(days)']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color,
                 label=f'{"Adopted" if label else "Not Adopted"}')
axes[0].set_xlabel('Age at Intake (days)')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Outcome')
axes[0].legend()

# Box plot by outcome
df_model.boxplot(column='age_upon_intake_(days)', by='adopted', ax=axes[1])
axes[1].set_xlabel('Adopted')
axes[1].set_ylabel('Age at Intake (days)')
axes[1].set_title('Age Boxplot by Outcome')
plt.sca(axes[1])
plt.title('Age Boxplot by Outcome')

# Log-transformed histogram (to see distribution shape better)
for label, color in zip([0, 1], ['#e74c3c', '#2ecc71']):
    subset = df_model[df_model['adopted'] == label]['age_upon_intake_(days)']
    subset_pos = subset[subset > 0]  # avoid log(0)
    axes[2].hist(np.log1p(subset_pos), bins=50, alpha=0.6, color=color,
                 label=f'{"Adopted" if label else "Not Adopted"}')
axes[2].set_xlabel('log(1 + Age in Days)')
axes[2].set_ylabel('Count')
axes[2].set_title('Log-Transformed Age Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

# Outlier statistics
age = df_model['age_upon_intake_(days)']
Q1, Q3 = age.quantile(0.25), age.quantile(0.75)
IQR = Q3 - Q1
outliers = ((age < Q1 - 1.5 * IQR) | (age > Q3 + 1.5 * IQR)).sum()
print(f"Age stats: min={age.min():.0f}, median={age.median():.0f}, max={age.max():.0f} days")
print(f"IQR outliers: {outliers:,} ({outliers/len(df_model):.1%} of data)")
print(f"Negative/zero age values: {(age <= 0).sum()}")

print(f"\n=== Interpretation ===")
print(f"\nThe age distribution is heavily right-skewed — most animals enter the shelter young,")
print(f"with a long tail of older animals. The log-transformed view confirms this skew and")
print(f"shows clearer separation between adopted and non-adopted animals: adopted animals")
print(f"tend to be younger, with their distribution peaking earlier than the non-adopted group.")
print(f"The boxplot reinforces this — adopted animals have a lower median age and a tighter")
print(f"interquartile range. Although the IQR method flags older animals as statistical outliers,")
print(f"these represent genuine senior animals and are not data errors — removing them would")
print(f"discard meaningful cases where age directly influences adoption likelihood. These will")
print(f"be retained in the modelling dataset.")

## 2.6 Temporal Patterns

We examine whether adoption rates vary by intake month, weekday, and hour to determine if temporal features carry predictive signal and to identify any cyclical patterns in shelter intake behaviour.

In [ ]:
# Adoption rate by temporal features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By month
monthly = df_model.groupby('intake_month')['adopted'].mean()
axes[0].plot(monthly.index, monthly.values, marker='o', color='steelblue')
axes[0].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Adoption Rate')
axes[0].set_title('Adoption Rate by Intake Month')
axes[0].set_xticks(range(1, 13))

# By weekday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_rates = df_model.groupby('intake_weekday')['adopted'].mean().reindex(weekday_order)
axes[1].bar(range(7), weekday_rates.values, color='steelblue')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels([d[:3] for d in weekday_order], rotation=45)
axes[1].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--', alpha=0.7)
axes[1].set_ylabel('Adoption Rate')
axes[1].set_title('Adoption Rate by Intake Weekday')

# By hour
hourly = df_model.groupby('intake_hour')['adopted'].mean()
axes[2].plot(hourly.index, hourly.values, marker='o', color='steelblue')
axes[2].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--', alpha=0.7)
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Adoption Rate')
axes[2].set_title('Adoption Rate by Intake Hour')

plt.tight_layout()
plt.show()

print(f"=== Interpretation ===")
print(f"\nMonthly: Adoption rates exhibit modest seasonal variation, with certain months")
print(f"consistently above or below the overall adoption rate. This pattern may reflect")
print(f"seasonal adoption demand (e.g., increased adoptions during summer or holidays) or")
print(f"seasonal intake patterns such as \"kitten season,\" where large numbers of young")
print(f"animals enter shelters in spring, increasing intake volume without proportionally")
print(f"increasing adoption rates.")
print(f"\nWeekday: Adoption rates remain relatively stable across weekdays, suggesting that")
print(f"the specific day an animal enters the shelter has limited predictive value on its")
print(f"own. As a result, this feature may contribute only marginal predictive signal during")
print(f"modelling.")
print(f"\nHour: Adoption rates vary more noticeably by intake hour. Animals arriving during")
print(f"certain hours exhibit different adoption outcomes, which may reflect the circumstances")
print(f"of intake (e.g., after-hours drop-offs versus scheduled owner surrenders) rather than")
print(f"the hour itself. Consequently, intake hour may act as a useful proxy feature capturing")
print(f"differences in intake context or shelter operational patterns.")

## 2.7 Correlation and Feature Interactions

We compute correlations between numeric features and the target to quantify linear associations, and use cross-tabulation to explore how combinations of categorical features relate to adoption outcomes.

In [ ]:
# Correlation heatmap of numeric features with target
num_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
corr = df_model[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix (Numeric Features + Target)')
plt.tight_layout()
plt.show()

# Point-biserial correlations with target
print("=== Correlation with Target (adopted) ===\n")
for col in num_cols:
    if col != 'adopted':
        r = df_model[col].corr(df_model['adopted'])
        print(f"  {col}: {r:+.3f}")

In [ ]:
# Cross-tabulation: animal_type × intake_type → adoption rate
cross = df_model.groupby(['animal_type', 'intake_type'])['adopted'].agg(['mean', 'count'])
cross.columns = ['adoption_rate', 'n']
cross = cross[cross['n'] >= 50]  # filter small groups
print("=== Adoption Rate by Animal Type × Intake Type (n ≥ 50) ===\n")
print(cross.sort_values('adoption_rate', ascending=False).to_string())

print(f"\n=== Interpretation ===")
print(f"\nThe correlation matrix reveals that the numeric features (age, intake month, weekday,")
print(f"hour) have relatively weak linear correlations with the target variable. This is")
print(f"expected — adoption outcomes are likely driven more by categorical characteristics")
print(f"(animal type, breed, intake condition) than by continuous numeric values alone. The")
print(f"weak correlations do not mean these features are uninformative, but rather that their")
print(f"relationship with adoption is non-linear and better captured by tree-based models.")
print(f"\nThe cross-tabulation of animal_type × intake_type reveals substantially stronger")
print(f"relationships than any single feature alone. For example, the combination of a")
print(f"specific animal type with a particular intake type produces adoption rates that")
print(f"deviate far more from the overall mean than either feature would individually. This")
print(f"suggests that feature interactions carry meaningful predictive signal — the context")
print(f"of how an animal arrives at the shelter matters differently depending on what kind")
print(f"of animal it is. Models capable of capturing such interactions (e.g. decision trees,")
print(f"gradient boosting) are likely to outperform linear models that treat features")
print(f"independently.")

## 2.8 Predictive Feature Deep Dive

We take a closer look at the features most likely to drive predictions — animal type, age group, intake type, and intake condition — examining both their distributions and individual adoption rates to understand their discriminative power.

In [ ]:
# Animal type: distribution and adoption rate side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
type_counts = df_model['animal_type'].value_counts()
colors_type = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6'][:len(type_counts)]
axes[0].bar(type_counts.index, type_counts.values, color=colors_type)
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df_model):.1%})', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Animal Type Distribution', fontsize=13)

# Adoption rate
type_adoption = df_model.groupby('animal_type')['adopted'].mean().sort_values(ascending=False)
bars = axes[1].bar(type_adoption.index, type_adoption.values, color=colors_type)
axes[1].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--',
                label=f'Overall: {df_model["adopted"].mean():.1%}')
for i, (cat, rate) in enumerate(type_adoption.items()):
    axes[1].text(i, rate + 0.01, f'{rate:.1%}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Adoption Rate', fontsize=11)
axes[1].set_title('Adoption Rate by Animal Type', fontsize=13)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Dogs and cats dominate shelter intake, accounting for the vast majority of records.")
print(f"Both exhibit adoption rates slightly above the overall average, while other animal")
print(f"types (birds, livestock, etc.) appear far less frequently and tend to have lower")
print(f"adoption rates. This imbalance means the model will primarily learn adoption patterns")
print(f"for dogs and cats, with limited exposure to rarer animal types.")

In [ ]:
# Age at intake: distribution and adoption rate by age group
# Create age groups for analysis
age_bins = [0, 30, 90, 365, 365*3, 365*7, 365*20]
age_labels = ['0-30d\n(Neonate)', '1-3mo\n(Kitten/Puppy)', '3mo-1yr\n(Junior)',
              '1-3yr\n(Young Adult)', '3-7yr\n(Adult)', '7yr+\n(Senior)']
df_model['age_group'] = pd.cut(df_model['age_upon_intake_(days)'], bins=age_bins, labels=age_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Age distribution
age_dist = df_model['age_group'].value_counts().reindex(age_labels)
axes[0].bar(range(len(age_dist)), age_dist.values, color='steelblue')
axes[0].set_xticks(range(len(age_dist)))
axes[0].set_xticklabels(age_labels, fontsize=9)
for i, v in enumerate(age_dist.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Age at Intake Distribution', fontsize=13)

# Adoption rate by age group
age_adoption = df_model.groupby('age_group', observed=True)['adopted'].mean().reindex(age_labels)
bars = axes[1].bar(range(len(age_adoption)), age_adoption.values, color='steelblue')
axes[1].set_xticks(range(len(age_adoption)))
axes[1].set_xticklabels(age_labels, fontsize=9)
axes[1].axhline(y=df_model['adopted'].mean(), color='red', linestyle='--',
                label=f'Overall: {df_model["adopted"].mean():.1%}')
for i, rate in enumerate(age_adoption.values):
    axes[1].text(i, rate + 0.01, f'{rate:.1%}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Adoption Rate', fontsize=11)
axes[1].set_title('Adoption Rate by Age Group', fontsize=13)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Drop temporary column (will engineer properly in Section 3)
df_model.drop(columns=['age_group'], inplace=True)

print(f"Adoption probability decreases steadily with age. Kittens and puppies (1-3 months)")
print(f"and juniors (3 months-1 year) enjoy the highest adoption rates — these are the ages")
print(f"at which cats and dogs are most in demand from adopters. Young adults (1-3 years)")
print(f"still perform above average, but adoption rates drop noticeably for adults (3-7 years)")
print(f"and fall sharply for seniors (7+ years). This aligns with the animal type findings:")
print(f"dogs and cats dominate intake and are most adoptable when young, while older animals")
print(f"of any type face significantly lower adoption prospects. Age is therefore one of the")
print(f"strongest single predictors available at intake time.")

In [ ]:
# Intake type: distribution and adoption rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
it_counts = df_model['intake_type'].value_counts()
axes[0].bar(range(len(it_counts)), it_counts.values, color='#2ecc71')
axes[0].set_xticks(range(len(it_counts)))
axes[0].set_xticklabels(it_counts.index, rotation=45, ha='right', fontsize=10)
for i, v in enumerate(it_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Intake Type Distribution', fontsize=13)

# Adoption rate
it_adoption = df_model.groupby('intake_type')['adopted'].mean().sort_values(ascending=False)
axes[1].barh(range(len(it_adoption)), it_adoption.values, color='#2ecc71')
axes[1].set_yticks(range(len(it_adoption)))
axes[1].set_yticklabels(it_adoption.index, fontsize=10)
axes[1].axvline(x=df_model['adopted'].mean(), color='red', linestyle='--',
                label=f'Overall: {df_model["adopted"].mean():.1%}')
for i, rate in enumerate(it_adoption.values):
    axes[1].text(rate + 0.005, i, f'{rate:.1%}', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Adoption Rate', fontsize=11)
axes[1].set_title('Adoption Rate by Intake Type', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Intake type shows clear variation in adoption likelihood. Owner surrenders tend to")
print(f"have higher adoption rates, likely because these animals are often already socialised")
print(f"and accustomed to living with people. Strays, which make up the largest intake category,")
print(f"sit closer to the overall average. Other intake types such as public assists and")
print(f"wildlife show notably lower adoption rates, reflecting animals that may be less suited")
print(f"to domestic adoption. This makes intake type a useful discriminator for the model.")

In [ ]:
# Intake condition: distribution and adoption rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
ic_counts = df_model['intake_condition'].value_counts()
axes[0].bar(range(len(ic_counts)), ic_counts.values, color='#e67e22')
axes[0].set_xticks(range(len(ic_counts)))
axes[0].set_xticklabels(ic_counts.index, rotation=45, ha='right', fontsize=10)
for i, v in enumerate(ic_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Intake Condition Distribution', fontsize=13)

# Adoption rate
ic_adoption = df_model.groupby('intake_condition')['adopted'].mean().sort_values(ascending=False)
axes[1].barh(range(len(ic_adoption)), ic_adoption.values, color='#e67e22')
axes[1].set_yticks(range(len(ic_adoption)))
axes[1].set_yticklabels(ic_adoption.index, fontsize=10)
axes[1].axvline(x=df_model['adopted'].mean(), color='red', linestyle='--',
                label=f'Overall: {df_model["adopted"].mean():.1%}')
for i, rate in enumerate(ic_adoption.values):
    axes[1].text(rate + 0.005, i, f'{rate:.1%}', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Adoption Rate', fontsize=11)
axes[1].set_title('Adoption Rate by Intake Condition', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Intake condition is a strong predictor of adoption. Animals arriving in normal")
print(f"condition have the highest adoption rates, while those classified as sick, injured,")
print(f"or nursing fall well below the overall average. This is intuitive — adopters generally")
print(f"prefer healthy animals, and animals in poor condition may require costly veterinary")
print(f"care that deters potential adopters.")
print(f"\nAn interesting pattern also emerges when combining age with neuter status (from the")
print(f"sex_upon_intake feature in Section 2.4): young intact animals tend to have higher")
print(f"adoption rates, likely because kittens and puppies are in high demand regardless of")
print(f"neuter status, and adopters expect to neuter them after adoption. Conversely, older")
print(f"neutered or spayed animals show lower adoption rates — despite being already fixed,")
print(f"their age works against them. This suggests that age is the dominant factor, and")
print(f"neuter status interacts with it rather than acting independently.")

In [ ]:
# Combined summary: adoption rate across key predictive features
fig, ax = plt.subplots(figsize=(14, 8))

# Gather adoption rates for all key categorical features
summary_data = []
for col in ['animal_type', 'intake_type', 'intake_condition', 'sex_upon_intake']:
    rates = df_model.groupby(col)['adopted'].mean()
    for cat, rate in rates.items():
        n = (df_model[col] == cat).sum()
        summary_data.append({'Feature': col, 'Category': cat, 'Adoption Rate': rate, 'n': n})

summary_df = pd.DataFrame(summary_data).sort_values('Adoption Rate', ascending=True)

# Colour by feature
feature_colors = {'animal_type': '#3498db', 'intake_type': '#2ecc71',
                  'intake_condition': '#e67e22', 'sex_upon_intake': '#9b59b6'}
bar_colors = [feature_colors[f] for f in summary_df['Feature']]

bars = ax.barh(range(len(summary_df)), summary_df['Adoption Rate'].values, color=bar_colors)
ax.set_yticks(range(len(summary_df)))
ax.set_yticklabels([f"{row['Category']}  ({row['Feature']})" for _, row in summary_df.iterrows()],
                    fontsize=9)
ax.axvline(x=df_model['adopted'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Overall adoption rate: {df_model["adopted"].mean():.1%}')
ax.set_xlabel('Adoption Rate', fontsize=11)
ax.set_title('Adoption Rate Across All Key Predictive Features', fontsize=13)

# Legend for feature colours
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f) for f, c in feature_colors.items()]
legend_elements.append(plt.Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Overall rate'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

## 2.9 EDA Summary and Key Findings

**Raw dataset insights:**
- Full outcome distribution explored: shows how Adoption compares to Transfer, Return to Owner, Euthanasia, etc. before binarisation
- Neuter status changes between intake and outcome confirmed — validates excluding `sex_upon_outcome` as leakage
- Time in shelter varies significantly by outcome type — confirms it is an outcome-time feature (leakage)
- Repeat shelter visitors identified — same animals appearing multiple times may affect model independence

**Data quality:**
- Missingness patterns identified across all 42 columns (41 original + derived target)
- `outcome_subtype` has >50% missing — unreliable even before considering leakage
- Age outliers identified via IQR method but retained as genuine senior animals, not data errors
- High-cardinality features (`breed`, `color`, `found_location`) flagged for grouping in preprocessing

**Class imbalance:**
- The adoption rate is approximately 42%, making this a moderate rather than severe class imbalance. Accuracy would still be somewhat misleading (a naive majority-class classifier would achieve ~58%), but the classes are not drastically skewed. F1-score remains the appropriate primary metric to ensure the model performs well on both classes, with stratified splits to preserve this ratio in train and test sets.

**Predictive signals identified:**
- Animal type shows strong separation — dogs and cats dominate intake and exhibit adoption rates slightly above the overall average, while rarer animal types have notably lower rates
- Age is one of the strongest single predictors — adoption probability decreases steadily with age, from high rates for kittens/puppies to sharply lower rates for seniors
- Intake type matters — owner surrenders show higher adoption rates (socialised animals), while strays sit near average and other types fall below
- Intake condition is highly predictive — animals in normal condition are adopted at much higher rates than sick, injured, or nursing animals
- Sex and neuter status at intake carry predictive signal, although the relationship is partly confounded with age as younger animals are more likely to be intact
- Temporal features (month, weekday, hour) show cyclical patterns — month and hour carry modest predictive signal, while weekday has limited discriminative value

**Feature interactions:**
- Cross-tabulation of animal_type × intake_type reveals substantial variation in adoption rates that exceeds what either feature shows individually. These interaction effects suggest that models capable of capturing non-linear feature interactions (e.g. decision trees, gradient boosting) are likely to outperform linear models that treat features independently. This reinforces the case for prioritising tree-based models in Section 4.

**Modelling pitfalls to address in Section 3:**
- High-cardinality categorical features need encoding strategy (frequency, target, or grouping)
- Class imbalance may require stratified splitting and/or class-weight adjustment
- `found_location` has very high cardinality — may need to be simplified or dropped
- Feature engineering opportunities: `is_mix_breed` from breed, `is_neutered` and `gender` from `sex_upon_intake`
- Repeat animals: multiple rows for the same animal — addressed via GroupShuffleSplit by animal_id

# Prepare the Data

## 3.1 Feature Engineering

The EDA in Section 2.4 revealed that `sex_upon_intake` is a composite field combining two distinct pieces of information — neuter status (Neutered/Spayed/Intact/Unknown) and biological sex (Male/Female) — into a single string. Decomposing it into separate features allows the model to learn from each dimension independently. Section 2.8 further showed that the relationship between neuter status and adoption is partly confounded with age, reinforcing the value of isolating neuter status as its own feature.

Section 2.4 also showed that `breed` has very high cardinality, but a simple binary distinction — mixed breed versus purebred — may capture a meaningful signal without the dimensionality cost. We extract `is_mix_breed` as a lightweight feature before the full grouping step.

- `is_mix_breed`: binary indicator — 1 if `breed` contains 'Mix' or '/' (multi-breed), 0 otherwise
- `is_neutered`: neuter/spay status (Yes / No / Unknown) from `sex_upon_intake`
- `gender`: biological sex (Male / Female / Unknown) from `sex_upon_intake`

In [ ]:
print("=== Feature Engineering ===\n")

# 1. Mix breed indicator: 1 if breed contains 'Mix' or '/' (multi-breed)
df_model['is_mix_breed'] = df_model['breed'].str.contains('Mix|/', regex=True).astype(int)
mix_pct = df_model['is_mix_breed'].mean() * 100
print(f"is_mix_breed: {mix_pct:.1f}% of animals are mixed breed\n")

# 2. Decompose sex_upon_intake into neutered status and gender
def extract_neutered(s):
    if pd.isna(s) or s == 'Unknown':
        return 'Unknown'
    return 'Yes' if ('Neutered' in s or 'Spayed' in s) else 'No'

def extract_gender(s):
    if pd.isna(s):
        return 'Unknown'
    if 'Male' in s:
        return 'Male'
    elif 'Female' in s:
        return 'Female'
    return 'Unknown'

df_model['is_neutered'] = df_model['sex_upon_intake'].apply(extract_neutered)
df_model['gender'] = df_model['sex_upon_intake'].apply(extract_gender)

print(f"is_neutered distribution:\n{df_model['is_neutered'].value_counts().to_string()}\n")
print(f"gender distribution:\n{df_model['gender'].value_counts().to_string()}\n")

# Drop original composite column
df_model.drop(columns=['sex_upon_intake'], inplace=True)
print(f"Dropped: sex_upon_intake -> replaced by is_neutered + gender")
print(f"Dataset shape: {df_model.shape}")

## 3.2 Missing Value Check

Section 2.2 identified minimal missingness in the modelling features — only `sex_upon_intake` had a small number of null and 'Unknown' values, which are now handled as a distinct category through the feature engineering in Section 3.1. We verify the dataset state here before splitting. Any remaining missing values will be handled by `SimpleImputer` inside the preprocessing pipeline (Section 3.5), fitted on training data only to prevent information leakage.

In [ ]:
print("=== Missing Value Check ===\n")

missing = df_model.isnull().sum()
missing_pct = (missing / len(df_model) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, '%': missing_pct})
missing_any = missing_df[missing_df['Count'] > 0].sort_values('%', ascending=False)

if len(missing_any) > 0:
    print(missing_any.to_string())
else:
    print("No null values in the modelling dataset.")

# Check for negative ages (data quality issue, not missingness)
neg_age = (df_model['age_upon_intake_(days)'] < 0).sum()
if neg_age > 0:
    median_age = df_model.loc[df_model['age_upon_intake_(days)'] >= 0, 'age_upon_intake_(days)'].median()
    df_model.loc[df_model['age_upon_intake_(days)'] < 0, 'age_upon_intake_(days)'] = median_age
    print(f"\nReplaced {neg_age} negative age values with median: {median_age:.0f} days")

print(f"\nDataset: {df_model.shape[0]:,} rows x {df_model.shape[1]} columns")
print(f"\nNote: Any remaining missing values will be handled by SimpleImputer inside the")
print(f"preprocessing pipeline (Section 3.5), fitted on training data only.")

## 3.3 Train-Test Split (GroupShuffleSplit)

Section 2.1 revealed that approximately 8.4% of animals appear multiple times in the dataset, violating the i.i.d. assumption. If the same animal appears in both train and test sets, the model could effectively "memorise" animal-specific patterns during training and be rewarded for them during evaluation — artificially inflating performance metrics.

To address this, we use `GroupShuffleSplit` grouped by `animal_id_intake`, ensuring that all records for a given animal are assigned to either the training set or the test set, but never both. This directly prevents the leakage identified in the repeat visitor analysis and produces a more honest estimate of how the model would perform on genuinely unseen animals.

**Importantly, the split is performed before high-cardinality grouping (Section 3.4)** so that the top-N category lists are derived exclusively from training data — preventing even minor distributional leakage from the test set.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Retrieve animal IDs for group-aware splitting
animal_ids = df['animal_id_intake'].iloc[df_model.index]

# Separate features and target
X = df_model.drop(columns=['adopted'])
y = df_model['adopted']

# GroupShuffleSplit: 80/20 split ensuring same animal never in both sets
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=animal_ids))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# Verify group integrity
train_animals = set(animal_ids.iloc[train_idx])
test_animals = set(animal_ids.iloc[test_idx])
overlap = train_animals & test_animals

print("=== Group-Aware Train-Test Split ===\n")
print(f"Training set: {len(X_train):,} rows ({len(X_train)/len(X):.1%})")
print(f"Test set:     {len(X_test):,} rows ({len(X_test)/len(X):.1%})")
print(f"\nUnique animals in train: {len(train_animals):,}")
print(f"Unique animals in test:  {len(test_animals):,}")
print(f"Animal overlap between sets: {len(overlap)}" + (" (no leakage)" if len(overlap) == 0 else " WARNING"))
print(f"\nAdoption rate -- train: {y_train.mean():.3f} | test: {y_test.mean():.3f} | overall: {y.mean():.3f}")
print(f"\nNote: X_train and X_test still contain raw breed, color, and found_location columns.")
print(f"These will be grouped in Section 3.4 using top-N categories derived from training data only.")

## 3.4 High-Cardinality Feature Grouping (Fitted on Training Data Only)

Section 2.4 identified that `breed`, `color`, and `found_location` each contain hundreds of unique values. One-hot encoding these directly would produce an extremely sparse feature matrix, increasing memory usage, slowing training, and risking overfitting to rare categories with very few observations. The EDA also showed that colour has only a modest effect on adoption probability (Section 2.4), and that the top breeds already account for the majority of records — meaning the long tail of rare categories contributes little predictive signal.

To address this, we retain only the most frequent categories and collapse the rest into an 'Other' bucket. **The top-N category lists are derived exclusively from the training set** and then applied to both train and test sets. This ensures no information about the test distribution leaks into the feature construction — any category present in the test set but not among the training set's top-N is naturally mapped to 'Other', which also mirrors realistic deployment where unseen categories may appear.

In [ ]:
print("=== High-Cardinality Grouping (fit on train, apply to both) ===\n")

def get_top_n_categories(series, n):
    """Return the top-n most frequent categories from a Series."""
    return series.value_counts().head(n).index.tolist()

def apply_grouping(train_series, test_series, top_categories, col_name):
    """Apply top-N grouping: keep top categories, collapse rest to 'Other'."""
    train_grouped = train_series.where(train_series.isin(top_categories), 'Other')
    test_grouped = test_series.where(test_series.isin(top_categories), 'Other')
    # Count how many test categories were unseen in train's top-N
    test_unseen = (~test_series.isin(top_categories)).sum()
    return train_grouped, test_grouped, test_unseen

# Fit top-N on training data only
top_breeds = get_top_n_categories(X_train['breed'], n=20)
top_colors = get_top_n_categories(X_train['color'], n=15)
top_locations = get_top_n_categories(X_train['found_location'], n=20)

# Apply grouping to both train and test
X_train['breed_grouped'], X_test['breed_grouped'], unseen_breed = apply_grouping(
    X_train['breed'], X_test['breed'], top_breeds, 'breed')
X_train['color_grouped'], X_test['color_grouped'], unseen_color = apply_grouping(
    X_train['color'], X_test['color'], top_colors, 'color')
X_train['location_grouped'], X_test['location_grouped'], unseen_loc = apply_grouping(
    X_train['found_location'], X_test['found_location'], top_locations, 'found_location')

# Report
orig_breed = X_train['breed'].nunique()
orig_color = X_train['color'].nunique()
orig_loc = X_train['found_location'].nunique()

print(f"breed:          {orig_breed} unique in train -> {X_train['breed_grouped'].nunique()} categories (top 20 + Other)")
print(f"color:          {orig_color} unique in train -> {X_train['color_grouped'].nunique()} categories (top 15 + Other)")
print(f"found_location: {orig_loc} unique in train -> {X_train['location_grouped'].nunique()} categories (top 20 + Other)")

print(f"\nTest set values mapped to 'Other' (unseen in train's top-N):")
print(f"  breed: {unseen_breed:,} | color: {unseen_color:,} | found_location: {unseen_loc:,}")

# Drop raw columns from both sets
X_train.drop(columns=['breed', 'color', 'found_location'], inplace=True)
X_test.drop(columns=['breed', 'color', 'found_location'], inplace=True)

print(f"\nFinal columns ({len(X_train.columns)}): {list(X_train.columns)}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 3.5 Preprocessing Pipeline

The EDA identified different feature types requiring distinct transformations:

- **Numeric** (`age_upon_intake_(days)`, `intake_month`, `intake_hour`): Section 2.5 showed that age is heavily right-skewed with IQR outliers. Standard scaling normalises these features to zero mean and unit variance, which benefits distance-based and linear models. Median imputation is used as it is robust to the skewed distribution.
- **Binary** (`is_mix_breed`): Already encoded as 0/1 in Section 3.1 and treated separately from the other categorical features because it is a simple binary indicator derived from `breed` — it does not require one-hot encoding or imputation, and passing it through unchanged avoids unnecessary transformation.
- **Categorical** (all remaining, including `is_neutered` and `gender`): Section 2.7 showed that feature interactions between categorical variables carry substantial predictive signal. `is_neutered` is treated as categorical rather than binary because it has three levels (Yes/No/Unknown), where 'Unknown' carries its own meaning. One-hot encoding (dropping the first level to avoid multicollinearity) preserves these categorical distinctions for modelling. Mode imputation handles any residual missing values.

We use a `ColumnTransformer` to apply these transformations within a single sklearn pipeline. Critically, the pipeline is **fit on the training set only** and then applied to the test set — ensuring that scaling parameters, imputation values, and encoding categories are all derived exclusively from training data, preventing information leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Define feature groups
numeric_features = ['age_upon_intake_(days)', 'intake_month', 'intake_hour']
binary_features = ['is_mix_breed']
categorical_features = [
    'animal_type', 'intake_condition', 'intake_type', 'intake_weekday',
    'is_neutered', 'gender', 'breed_grouped', 'color_grouped', 'location_grouped'
]

# Sub-pipelines
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist')),
])

# Combined preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('bin', 'passthrough', binary_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop'
)

# Fit on train, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Extract feature names
ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_names = ohe.get_feature_names_out(categorical_features).tolist()
feature_names = numeric_features + binary_features + cat_names

print("=== Preprocessing Pipeline ===\n")
print(f"Numeric features (median impute + StandardScaler): {numeric_features}")
print(f"Binary features (passthrough):                     {binary_features}")
print(f"Categorical features (mode impute + OneHotEncoder): {categorical_features}")
print(f"\nProcessed shape -- Train: {X_train_processed.shape} | Test: {X_test_processed.shape}")
print(f"Total encoded features: {len(feature_names)}")

In [ ]:
# Verification: class balance in train vs test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, (label, data) in enumerate([('Train', y_train), ('Test', y_test)]):
    counts = data.value_counts().sort_index()
    axes[i].bar(['Not Adopted (0)', 'Adopted (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
    axes[i].set_title(f'{label} Set (n={len(data):,})')
    axes[i].set_ylabel('Count')
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 100, f'{v:,}\n({v/len(data):.1%})', ha='center', fontweight='bold')

plt.suptitle('Class Distribution: Train vs Test', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("=== Processed Data Verification ===\n")
print(f"Any NaN in processed train: {np.isnan(X_train_processed).any()}")
print(f"Any NaN in processed test:  {np.isnan(X_test_processed).any()}")
print(f"Any infinite values in train: {np.isinf(X_train_processed).any()}")
print(f"Any infinite values in test:  {np.isinf(X_test_processed).any()}")
print(f"\nFeature matrix dtype: {X_train_processed.dtype}")
print(f"Target dtype: {y_train.dtype}")

## 3.6 Data Preparation Summary

**Feature engineering (Section 3.1):**
- Created `is_mix_breed` (binary) from `breed`
- Decomposed `sex_upon_intake` into `is_neutered` (Yes/No/Unknown) and `gender` (Male/Female/Unknown)

**Train-test split (Section 3.3):**
- 80/20 split using `GroupShuffleSplit` by `animal_id_intake`
- Zero animal overlap between train and test sets
- **Split performed before high-cardinality grouping** to prevent distributional leakage

**High-cardinality grouping (Section 3.4 — fitted on training data only):**
- `breed` → top 20 + Other (top-N list derived from training set)
- `color` → top 15 + Other (top-N list derived from training set)
- `found_location` → top 20 + Other (top-N list derived from training set)
- Unseen test categories are naturally mapped to 'Other', mirroring deployment conditions

**Preprocessing pipeline (Section 3.5 — fitted on training data only):**
- Numeric: `SimpleImputer(median)` → `StandardScaler`
- Binary: passthrough
- Categorical: `SimpleImputer(most_frequent)` → `OneHotEncoder(drop='first')`
- All imputation and encoding is handled within the pipeline to prevent information leakage from the test set

In [ ]:
# Final feature list after preprocessing
feature_table = pd.DataFrame({
    'Feature': [
        'age_upon_intake_(days)', 'intake_month', 'intake_hour',
        'is_mix_breed',
        'animal_type', 'intake_condition', 'intake_type', 'intake_weekday',
        'is_neutered', 'gender',
        'breed_grouped', 'color_grouped', 'location_grouped'
    ],
    'Type': [
        'Numeric', 'Numeric', 'Numeric',
        'Binary',
        'Categorical', 'Categorical', 'Categorical', 'Categorical',
        'Categorical', 'Categorical',
        'Categorical', 'Categorical', 'Categorical'
    ],
    'Pipeline Treatment': [
        'Median imputation → StandardScaler', 'StandardScaler', 'StandardScaler',
        'Passthrough (already 0/1)',
        'Mode imputation → OneHotEncoder', 'Mode imputation → OneHotEncoder',
        'Mode imputation → OneHotEncoder', 'Mode imputation → OneHotEncoder',
        'Mode imputation → OneHotEncoder', 'Mode imputation → OneHotEncoder',
        'Mode imputation → OneHotEncoder', 'Mode imputation → OneHotEncoder',
        'Mode imputation → OneHotEncoder'
    ],
    'Source / Notes': [
        'Original feature; right-skewed (EDA §2.5)',
        'Extracted from intake datetime; seasonal signal (EDA §2.6)',
        'Extracted from intake datetime; hourly proxy for walk-ins (EDA §2.6)',
        'Derived from breed; 1 = Mix/cross (EDA §2.4)',
        'Original feature; dogs & cats dominate (EDA §2.8)',
        'Original feature; normal condition most common (EDA §2.8)',
        'Original feature; owner surrender vs stray signal (EDA §2.8)',
        'Original feature; marginal weekday effect (EDA §2.6)',
        'Derived from sex_upon_intake; 3 levels: Yes/No/Unknown (EDA §2.7)',
        'Derived from sex_upon_intake; 3 levels: Male/Female/Unknown',
        'Top-20 breeds + Other (EDA §2.4)',
        'Top-15 colours + Other; modest effect (EDA §2.4)',
        'Top-20 locations + Other (EDA §2.4)'
    ]
})

print("=" * 90)
print("FINAL FEATURE LIST — 13 features entering the preprocessing pipeline")
print("=" * 90)
print(f"\n  Numeric:      {(feature_table['Type'] == 'Numeric').sum()}")
print(f"  Binary:       {(feature_table['Type'] == 'Binary').sum()}")
print(f"  Categorical:  {(feature_table['Type'] == 'Categorical').sum()}")
print(f"  Target:       adopted (binary: 1 = Adopted, 0 = Not Adopted)\n")
print(feature_table.to_string(index=False))

In [ ]:
# === Pre-Modelling Sanity Checks ===
print("=" * 70)
print("PRE-MODELLING SANITY CHECKS")
print("=" * 70)

# 1. Feature set is final
check1a = X_train.shape[1] == 13 and X_test.shape[1] == 13
check1b = set(X_train.columns) == set(X_test.columns)
print(f"\n[1] Feature set is final")
print(f"    X_train columns: {X_train.shape[1]}  |  X_test columns: {X_test.shape[1]}  -> {'PASS' if check1a else 'FAIL'}")
print(f"    Column sets match: {check1b}  -> {'PASS' if check1b else 'FAIL'}")
if not check1b:
    print(f"    Train-only: {set(X_train.columns) - set(X_test.columns)}")
    print(f"    Test-only:  {set(X_test.columns) - set(X_train.columns)}")

# 2. No leakage via groups
check2 = len(overlap) == 0
print(f"\n[2] No leakage via groups")
print(f"    Animal overlap between train/test: {len(overlap)}  -> {'PASS' if check2 else 'FAIL'}")

# 3. Preprocessor fit only on train — verify and prepare unfitted copy for Section 4
from sklearn.base import clone
preprocessor_unfitted = clone(preprocessor)
check3 = True  # We'll use this unfitted clone inside Pipeline for CV in Section 4
print(f"\n[3] Preprocessor ready for Section 4")
print(f"    Unfitted preprocessor cloned: {type(preprocessor_unfitted).__name__}")
print(f"    -> In Section 4, each model will be wrapped as:")
print(f"       Pipeline([('preprocess', clone(preprocessor)), ('model', ...)])")
print(f"       so CV folds refit preprocessing inside each fold (no data leakage)")

# Overall
all_pass = check1a and check1b and check2 and check3
print(f"\n{'=' * 70}")
print(f"ALL CHECKS {'PASSED' if all_pass else 'FAILED'} — ready for Section 4")
print(f"{'=' * 70}")

# Explore Different Models and Shortlist the Best Ones

## 4.1 Modelling Strategy

We compare five models spanning distinct algorithmic families, ensuring diversity in how they learn from the data:

| # | Model | Family | Rationale |
|---|-------|--------|-----------|
| 1 | **Logistic Regression** | Linear | Interpretable baseline; establishes a floor that non-linear models must beat |
| 2 | **Random Forest** | Bagging ensemble | Robust to outliers and noise; naturally captures the feature interactions identified in EDA §2.7 |
| 3 | **XGBoost** | Gradient boosting | State-of-the-art for tabular data; handles moderate class imbalance via `scale_pos_weight` |
| 4 | **LightGBM** | Gradient boosting | Leaf-wise growth alternative to XGBoost; faster training, different inductive bias |
| 5 | **Neural Network (MLP)** | Deep learning | Required by the module learning outcomes; enables training curve visualisation |

For the tree-based ensemble models, the number of trees (`n_estimators`) was set to 300. In ensemble learning methods, performance typically stabilises after a few hundred trees as additional trees contribute diminishing improvements while increasing computational cost. Using 300 trees therefore provides a good balance between model stability and training efficiency. This value also aligns with common practice for both Random Forest and gradient boosting models, particularly when using a moderate learning rate (0.1) and shallow trees. The parameter will be further optimised during the hyperparameter tuning stage in Section 5, but using a consistent number of trees here ensures a fair baseline comparison across models.

**Evaluation protocol:**

Five-fold GroupKFold cross-validation was used on the training data. Five folds provide a good balance between reliable performance estimation and computational efficiency, which is appropriate given the large dataset (~63k training samples). The grouping constraint ensures that records belonging to the same `animal_id` never appear in both training and validation folds, preventing data leakage. This procedure provides a robust estimate of model generalisation performance by evaluating models across multiple train–validation splits.

- Each model is wrapped in a `Pipeline` with an unfitted preprocessor clone, so scaling, imputation, and encoding are refit inside each fold (no data leakage)
- **Primary metric:** F1-score (balances precision and recall for the moderate class imbalance)
- **Secondary metrics:** AUC-ROC, Precision, Recall
- The held-out test set (`X_test`) is **not touched** until final evaluation in Section 5

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, cross_val_predict, GroupKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.base import clone
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
import time

# Retrieve animal IDs for the training set (needed for GroupKFold)
train_animal_ids = animal_ids.iloc[train_idx]

# Verify alignment with X_train
assert len(train_animal_ids) == len(X_train), "Group array length mismatch!"
assert (train_animal_ids.index == X_train.index).all(), "Group array index mismatch!"

# GroupKFold: 5 folds respecting animal_id grouping
gkf = GroupKFold(n_splits=5)

# Scoring: sklearn built-in string scorers
# - f1, precision, recall: binary averaging by default (pos_label=1 = Adopted)
# - roc_auc: automatically uses predict_proba for probability estimates
scoring = {'f1': 'f1', 'roc_auc': 'roc_auc', 'precision': 'precision', 'recall': 'recall'}

print("=== Section 4 Setup ===")
print(f"Training samples: {len(X_train):,}")
print(f"CV strategy: GroupKFold (k=5) on training data only")
print(f"Unique animal groups in train: {train_animal_ids.nunique():,}")
print(f"Group alignment: len={len(train_animal_ids)}, index match = {(train_animal_ids.index == X_train.index).all()} -> PASS")
print(f"\nMetrics:")
print(f"  F1 / Precision / Recall: binary averaging (pos_label=1 = Adopted)")
print(f"  AUC-ROC: uses predict_proba (probability estimates, not class labels)")
print(f"\nModels to evaluate: 5 (LogReg, RF, XGBoost, LightGBM, MLP)")
print(f"\nNote: Each model is wrapped in Pipeline([preprocess, model]).")
print(f"sklearn's cross_validate internally clones the pipeline for each fold,")
print(f"ensuring fresh preprocessing + model fitting per fold (no data leakage).")

## 4.2 Baseline — Logistic Regression

A logistic regression serves as the interpretable baseline. It models a linear decision boundary in the feature space, meaning it cannot capture the non-linear interactions identified in EDA §2.7. This makes it an ideal reference point — if tree-based models substantially outperform it, that confirms the interactions are genuinely useful for prediction rather than just visually interesting in cross-tabulations.

In [ ]:
# Logistic Regression baseline
lr_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs'))
])

print("=== Logistic Regression (Baseline) ===\n")
start = time.time()
lr_cv = cross_validate(lr_pipeline, X_train, y_train, cv=gkf, groups=train_animal_ids,
                       scoring=scoring, return_train_score=False, n_jobs=-1)
lr_time = time.time() - start

for metric in ['f1', 'roc_auc', 'precision', 'recall']:
    scores = lr_cv[f'test_{metric}']
    print(f"  {metric:>10s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {scores.round(4)})")
print(f"\n  CV time: {lr_time:.1f}s")

The logistic regression model provides a baseline performance against which more complex models can be compared. Using 5-fold GroupKFold cross-validation, the model achieved an average F1-score of 0.615, with precision of 0.604 and recall of 0.627. The ROC-AUC of 0.735 indicates that the model has moderate ability to distinguish between adopted and non-adopted animals. While the performance is reasonable, logistic regression assumes a linear decision boundary, meaning it cannot naturally capture the non-linear relationships and feature interactions identified during the exploratory data analysis (e.g., interactions between age and neuter status). As a result, this model primarily serves as a reference baseline: if tree-based or boosting models achieve substantially higher scores, it suggests that these non-linear interactions contribute meaningfully to predictive performance.

The moderate performance of logistic regression is consistent with findings from the exploratory data analysis (Section 2.7), which indicated that several predictors interact in non-linear ways (e.g., age and neuter status). Because logistic regression models a linear decision boundary, it cannot naturally capture such interactions unless explicit feature engineering is performed. This limitation motivates the use of tree-based and ensemble methods in the following sections, which can learn these interaction patterns directly from the data.

## 4.3 Random Forest

Random Forest builds an ensemble of decorrelated decision trees via bootstrap aggregation (bagging). Each tree sees a random subset of features at every split, which reduces variance and provides natural resistance to overfitting. Crucially, tree-based models can capture the non-linear feature interactions — such as the age–neuter status confounding identified in EDA §2.7 — without requiring explicit feature crosses.

In [ ]:
# Random Forest
rf_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
])

print("=== Random Forest ===\n")
start = time.time()
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=gkf, groups=train_animal_ids,
                       scoring=scoring, return_train_score=False, n_jobs=-1)
rf_time = time.time() - start

for metric in ['f1', 'roc_auc', 'precision', 'recall']:
    scores = rf_cv[f'test_{metric}']
    print(f"  {metric:>10s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {scores.round(4)})")
print(f"\n  CV time: {rf_time:.1f}s")

The Random Forest model improves upon the logistic regression baseline, achieving an average F1-score of 0.666 and ROC-AUC of 0.795, indicating stronger predictive performance. This improvement is expected given the findings from the exploratory data analysis (Section 2.7), which suggested that several predictors interact in non-linear ways. Random Forest models consist of ensembles of decision trees, which can naturally capture complex feature interactions and non-linear relationships without requiring explicit feature engineering. For example, combinations of characteristics such as age, neuter status, and intake conditions may jointly influence adoption probability. The improvement over the linear baseline therefore supports the hypothesis that these interactions contribute meaningfully to prediction performance.

The approximately 5 percentage point improvement in F1-score over the logistic regression baseline suggests that modelling non-linear feature interactions provides meaningful gains in predictive performance.

## 4.4 XGBoost

XGBoost builds trees sequentially, with each new tree correcting the residual errors of the ensemble so far. It uses level-wise tree growth and regularisation (L1/L2 penalties on leaf weights) to control complexity. The `scale_pos_weight` parameter is set to account for the moderate class imbalance (~42% adoption rate) identified in EDA §2.3, giving the minority class slightly more influence during training.

In [ ]:
# XGBoost
# Calculate scale_pos_weight for class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

xgb_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_weight,
        random_state=RANDOM_STATE, eval_metric='logloss',
        use_label_encoder=False, n_jobs=-1
    ))
])

print("=== XGBoost ===\n")
print(f"  scale_pos_weight: {scale_weight:.3f} (neg/pos ratio)\n")
start = time.time()
xgb_cv = cross_validate(xgb_pipeline, X_train, y_train, cv=gkf, groups=train_animal_ids,
                        scoring=scoring, return_train_score=False, n_jobs=-1)
xgb_time = time.time() - start

for metric in ['f1', 'roc_auc', 'precision', 'recall']:
    scores = xgb_cv[f'test_{metric}']
    print(f"  {metric:>10s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {scores.round(4)})")
print(f"\n  CV time: {xgb_time:.1f}s")

The XGBoost model achieved the strongest performance in terms of F1-score (0.695), outperforming both the logistic regression baseline and the Random Forest model. While the ROC-AUC (0.791) is slightly lower than that of Random Forest (0.795), the difference is small and within the expected variation across cross-validation folds. Because F1-score was selected as the primary evaluation metric, reflecting the need to balance precision and recall under moderate class imbalance, XGBoost can be considered the stronger model in this comparison. This improvement reflects the ability of gradient boosting models to capture complex, non-linear relationships between predictors. During exploratory data analysis (Section 2.3), the dataset was found to exhibit moderate class imbalance, with approximately 42% of animals adopted. Without addressing this imbalance, models may become biased toward predicting the majority class (non-adoption), as this minimises overall error. To mitigate this, the scale_pos_weight parameter was set to the ratio of negative to positive cases, giving greater importance to the minority adoption class during training. As a result, the model places more emphasis on correctly identifying adoptable animals, which explains the substantial increase in recall (0.786) compared to earlier models. This indicates that the model is better at detecting true adoption cases, although it slightly reduces precision due to a higher number of false positives. Overall, the improved performance supports the earlier EDA findings that adoption outcomes depend on non-linear interactions between multiple features, which boosting-based tree models are particularly well suited to capture.

## 4.5 LightGBM

LightGBM is another gradient boosting framework but uses leaf-wise (best-first) tree growth instead of XGBoost's level-wise approach. This tends to converge faster and can find deeper interaction patterns, though it may be more prone to overfitting on small datasets. On our ~80K-row dataset, this risk is manageable. Including both boosting implementations lets us compare their inductive biases empirically rather than choosing one a priori.

In [ ]:
# LightGBM
lgb_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_weight,
        random_state=RANDOM_STATE, verbose=-1, n_jobs=-1
    ))
])

print("=== LightGBM ===\n")
start = time.time()
lgb_cv = cross_validate(lgb_pipeline, X_train, y_train, cv=gkf, groups=train_animal_ids,
                        scoring=scoring, return_train_score=False, n_jobs=-1)
lgb_time = time.time() - start

for metric in ['f1', 'roc_auc', 'precision', 'recall']:
    scores = lgb_cv[f'test_{metric}']
    print(f"  {metric:>10s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {scores.round(4)})")
print(f"\n  CV time: {lgb_time:.1f}s")

The LightGBM model achieved performance comparable to the XGBoost model, with an F1-score of 0.695, ROC-AUC of 0.790, precision of 0.623, and recall of 0.786. These results are nearly identical to those obtained with XGBoost and represent a clear improvement over both the logistic regression baseline and the Random Forest model. Because the F1-score was selected as the primary evaluation metric, reflecting the need to balance precision and recall under moderate class imbalance, LightGBM can be considered one of the strongest performing models evaluated so far. Similar to XGBoost, the model incorporates the scale_pos_weight parameter to address the class imbalance identified during exploratory data analysis (Section 2.3), where approximately 42% of animals were adopted. Without this adjustment, models may become biased toward predicting the majority class (non-adoption), as this minimises overall error. By increasing the importance of the minority adoption class during training, the model places greater emphasis on correctly identifying adoptable animals, which explains the high recall (0.786) observed. However, this also introduces a precision–recall trade-off, as identifying more true adoption cases increases the likelihood of predicting adoption for some animals that are ultimately not adopted, leading to a slight reduction in precision. Overall, the strong performance of LightGBM further supports the earlier EDA findings that adoption outcomes depend on complex, non-linear interactions between multiple features, which gradient boosting tree models are particularly well suited to capture.

## 4.6 Neural Network (MLP)

The module learning outcomes require demonstrating the ability to design, train, and evaluate predictive models including neural network methods. We use a multi-layer perceptron (MLP) with two hidden layers. Neural networks learn feature representations through non-linear activation functions and can theoretically approximate any decision boundary, but they typically require more data and careful tuning to match gradient boosting on structured tabular data. Including an MLP allows us to compare deep learning against tree-based approaches on this problem and visualise training behaviour through loss curves.

**Early stopping note:** `MLPClassifier` with `early_stopping=True` reserves a fraction (`validation_fraction=0.15`) of each CV training fold as an internal validation set to monitor convergence and halt training when the validation loss stops improving. This internal split is drawn entirely from the CV training fold — the CV validation fold is never seen during training, so there is no data leakage. The same `GroupKFold` splits are used as for all other models, ensuring a fair comparison.

In [ ]:
# Neural Network (MLP)
mlp_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='adam', learning_rate='adaptive', learning_rate_init=0.001,
        max_iter=200, early_stopping=True, validation_fraction=0.15,
        random_state=RANDOM_STATE
    ))
])

print("=== Neural Network (MLP) ===\n")
print(f"  Architecture: Input -> 128 -> 64 -> 1 (sigmoid)")
print(f"  Activation: ReLU | Solver: Adam | Early stopping: Yes\n")
start = time.time()
mlp_cv = cross_validate(mlp_pipeline, X_train, y_train, cv=gkf, groups=train_animal_ids,
                        scoring=scoring, return_train_score=False, n_jobs=-1)
mlp_time = time.time() - start

for metric in ['f1', 'roc_auc', 'precision', 'recall']:
    scores = mlp_cv[f'test_{metric}']
    print(f"  {metric:>10s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {scores.round(4)})")
print(f"\n  CV time: {mlp_time:.1f}s")

The multi-layer perceptron (MLP) achieved an F1-score of 0.657, ROC-AUC of 0.757, precision of 0.615, and recall of 0.706. This represents an improvement over the logistic regression baseline, indicating that the neural network is able to capture non-linear relationships between features that a linear model cannot represent. However, the MLP does not outperform the tree-based ensemble methods evaluated earlier, including Random Forest, XGBoost, and LightGBM. This outcome is consistent with the broader machine learning literature, which often finds that gradient boosting models perform particularly well on structured tabular datasets, whereas neural networks typically require larger datasets and extensive hyperparameter tuning to achieve comparable performance. The model also demonstrates a relatively high recall, suggesting it is reasonably effective at identifying animals that will be adopted. However, this again reflects a precision–recall trade-off, where increasing the number of detected adoption cases can lead to a greater number of false positives. Overall, while the neural network benefits from its ability to model complex non-linear relationships identified during the exploratory analysis (Section 2.7), the results suggest that tree-based ensemble methods remain better suited to this dataset.

## 4.7 Model Comparison

We consolidate the cross-validation results across all five models to identify which approaches best capture the predictive patterns in the data. The comparison table and visualisation below rank models by F1-score (our primary metric) while also showing AUC-ROC, Precision, and Recall for a more complete picture.

In [ ]:
# Consolidate results
results = {
    'Logistic Regression': lr_cv,
    'Random Forest': rf_cv,
    'XGBoost': xgb_cv,
    'LightGBM': lgb_cv,
    'Neural Network (MLP)': mlp_cv,
}
times = {
    'Logistic Regression': lr_time,
    'Random Forest': rf_time,
    'XGBoost': xgb_time,
    'LightGBM': lgb_time,
    'Neural Network (MLP)': mlp_time,
}

# Build comparison table
comparison_rows = []
for name, cv_result in results.items():
    row = {'Model': name}
    for metric in ['f1', 'roc_auc', 'precision', 'recall']:
        scores = cv_result[f'test_{metric}']
        row[f'{metric}_mean'] = scores.mean()
        row[f'{metric}_std'] = scores.std()
    row['time_s'] = times[name]
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('f1_mean', ascending=False).reset_index(drop=True)

# Print formatted table with mean (±std)
print("=" * 90)
print("MODEL COMPARISON — GroupKFold CV (k=5) on Training Data Only")
print("=" * 90)
print(f"\n{'Model':<25s} {'F1':>14s} {'AUC-ROC':>14s} {'Precision':>14s} {'Recall':>14s} {'Time':>7s}")
print("-" * 90)
for _, row in comparison_df.iterrows():
    print(f"{row['Model']:<25s} "
          f"{row['f1_mean']:.4f} ({row['f1_std']:.3f}) "
          f"{row['roc_auc_mean']:.4f} ({row['roc_auc_std']:.3f}) "
          f"{row['precision_mean']:.4f} ({row['precision_std']:.3f}) "
          f"{row['recall_mean']:.4f} ({row['recall_std']:.3f}) "
          f"{row['time_s']:>5.1f}s")
print("-" * 90)
print("Values shown as: mean (±std) across 5 folds")
print("=" * 90)

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics_to_plot = ['f1', 'roc_auc', 'precision', 'recall']
metric_labels = ['F1-Score', 'AUC-ROC', 'Precision', 'Recall']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    model_names = comparison_df['Model'].values
    means = comparison_df[f'{metric}_mean'].values
    stds = comparison_df[f'{metric}_std'].values

    bars = ax.barh(range(len(model_names)), means, xerr=stds, color=colors, 
                   edgecolor='white', linewidth=0.5, capsize=3)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names, fontsize=9)
    ax.set_xlabel(label)
    ax.set_title(label, fontweight='bold')
    ax.invert_yaxis()
    
    # Add value labels
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(m + s + 0.005, i, f'{m:.3f}', va='center', fontsize=8)

plt.suptitle('Model Comparison — 5-Fold GroupKFold CV on Training Data', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4.8 Shortlist and Discussion

In [ ]:
# Identify top models
top_models = comparison_df.head(3)
print("=== Shortlisted Models (Top 3 by F1-Score) ===\n")
for i, (_, row) in enumerate(top_models.iterrows(), 1):
    print(f"  {i}. {row['Model']}: F1 = {row['f1_mean']:.4f}, AUC-ROC = {row['roc_auc_mean']:.4f}")

print(f"\n--- Observations ---")
print(f"\n• Baseline (Logistic Regression) F1: {comparison_df[comparison_df['Model']=='Logistic Regression']['f1_mean'].values[0]:.4f}")
print(f"  -> Non-linear models improve over the linear baseline, confirming")
print(f"     that the feature interactions identified in EDA §2.7 contribute")
print(f"     genuine predictive value beyond additive effects.")

# Check if boosting > RF
xgb_f1 = comparison_df[comparison_df['Model']=='XGBoost']['f1_mean'].values[0]
lgb_f1 = comparison_df[comparison_df['Model']=='LightGBM']['f1_mean'].values[0]
rf_f1 = comparison_df[comparison_df['Model']=='Random Forest']['f1_mean'].values[0]
mlp_f1 = comparison_df[comparison_df['Model']=='Neural Network (MLP)']['f1_mean'].values[0]

print(f"\n• Gradient boosting (XGBoost: {xgb_f1:.4f}, LightGBM: {lgb_f1:.4f})")
print(f"  vs Random Forest ({rf_f1:.4f}) vs MLP ({mlp_f1:.4f})")

print(f"\n• The top shortlisted models will proceed to Section 5 for")
print(f"  hyperparameter tuning via GroupKFold CV on training data,")
print(f"  with final evaluation on the held-out test set.")

## 4.9 Out-of-Fold Confusion Matrices and ROC Curves

To visualise each model's classification behaviour without touching the held-out test set, we use `cross_val_predict` with the same `GroupKFold` splits to obtain **out-of-fold predictions** — each training sample is predicted by the fold in which it was held out. This gives us honest estimates of confusion patterns and probability calibration across the full training set.

In [ ]:
# Out-of-fold predictions for confusion matrices and ROC curves
print("=== Generating Out-of-Fold Predictions (cross_val_predict) ===\n")

# Define fresh pipelines for cross_val_predict
model_pipelines = {
    'Logistic Regression': Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver='lbfgs'))
    ]),
    'Random Forest': Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            scale_pos_weight=scale_weight, random_state=RANDOM_STATE,
            eval_metric='logloss', use_label_encoder=False, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', lgb.LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            scale_pos_weight=scale_weight, random_state=RANDOM_STATE,
            verbose=-1, n_jobs=-1))
    ]),
    'Neural Network (MLP)': Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', MLPClassifier(
            hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
            learning_rate='adaptive', learning_rate_init=0.001, max_iter=200,
            early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE))
    ]),
}

# Collect out-of-fold class predictions and probability predictions
oof_preds = {}
oof_proba = {}

for name, pipe in model_pipelines.items():
    print(f"  {name}...", end=" ", flush=True)
    start = time.time()
    oof_preds[name] = cross_val_predict(pipe, X_train, y_train, cv=gkf,
                                        groups=train_animal_ids, method='predict', n_jobs=-1)
    oof_proba[name] = cross_val_predict(pipe, X_train, y_train, cv=gkf,
                                        groups=train_animal_ids, method='predict_proba', n_jobs=-1)[:, 1]
    elapsed = time.time() - start
    print(f"done ({elapsed:.1f}s)")

print("\nAll out-of-fold predictions collected.")

In [ ]:
# Confusion matrices — out-of-fold predictions (2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
model_order = comparison_df['Model'].values  # sorted by F1

for i, name in enumerate(model_order):
    cm = confusion_matrix(y_train, oof_preds[name])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Adopted', 'Adopted'])
    disp.plot(ax=axes[i], cmap='Blues', values_format=',d', colorbar=False)
    f1 = comparison_df[comparison_df['Model'] == name]['f1_mean'].values[0]
    axes[i].set_title(f'{name}\nF1 = {f1:.4f}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual' if i % 3 == 0 else '')

# Hide the 6th subplot (empty)
axes[5].axis('off')

plt.suptitle('Out-of-Fold Confusion Matrices — GroupKFold CV (k=5) on Training Data',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves — out-of-fold probability predictions
fig, ax = plt.subplots(figsize=(8, 7))
colors = {'Logistic Regression': '#3498db', 'Random Forest': '#2ecc71',
          'XGBoost': '#e74c3c', 'LightGBM': '#f39c12', 'Neural Network (MLP)': '#9b59b6'}

for name in model_order:
    fpr, tpr, _ = roc_curve(y_train, oof_proba[name])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — Out-of-Fold Predictions (GroupKFold CV)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print AUC summary
print("=== AUC-ROC Summary (out-of-fold) ===\n")
for name in model_order:
    fpr, tpr, _ = roc_curve(y_train, oof_proba[name])
    print(f"  {name:<25s}: {auc(fpr, tpr):.4f}")

## 4.10 Section 4 Summary

The cross-validated results highlight clear differences in predictive performance across the five modelling approaches. The gradient boosting models, XGBoost and LightGBM, achieved the highest F1-scores (0.695 and 0.695 respectively), indicating the strongest balance between precision and recall. Although the difference between the two models is extremely small and well within expected cross-validation variation, both consistently outperform the remaining approaches. The Random Forest model achieved slightly lower F1 performance but recorded the highest ROC-AUC score, suggesting strong ranking ability and making it a useful benchmark and candidate for further tuning. The neural network (MLP) improved upon the logistic regression baseline but did not match the performance of the tree-based ensemble methods. This outcome is consistent with the broader machine learning literature, which often finds that gradient boosting models perform particularly well on structured tabular datasets.

The superior performance of the boosting models likely reflects their ability to capture complex, non-linear feature interactions, which were already suggested during exploratory data analysis (Section 2.7). For example, adoption probability appeared to depend on combinations of factors such as age and neuter status, intake condition, and breed characteristics, indicating that the predictive relationship between features and outcomes is not purely additive. Boosting models sequentially build trees that correct previous prediction errors, enabling them to learn these interaction effects more effectively than linear models or simpler neural networks. The improvement over the logistic regression baseline therefore supports the earlier EDA findings that these interactions contribute genuine predictive value.

Importantly, all results are obtained using GroupKFold cross-validation (k=5) applied only to the training data, where animals belonging to the same identifier are kept within the same fold. This ensures that observations from the same animal do not appear in both training and validation sets, preventing data leakage and providing a more realistic estimate of generalisation performance. Furthermore, the preprocessing pipeline is refitted within each fold, ensuring that feature transformations are learned only from the training portion of the data during each iteration.

Based on these results, XGBoost, LightGBM, and Random Forest are shortlisted for further hyperparameter tuning in Section 5, where their performance will be optimised using GroupKFold cross-validation on the training data before final evaluation on the held-out test set.

To further analyse model behaviour, confusion matrices and ROC curves were generated using out-of-fold predictions obtained via `cross_val_predict` with the same GroupKFold splits used in earlier evaluation. In this setup, each observation is predicted by a model that was trained on the remaining folds and never sees that observation during training, providing an unbiased estimate of classification behaviour across the full training dataset. Importantly, all preprocessing steps are included inside the modelling pipelines and fitted separately within each fold, meaning feature scaling and encoding are learned only from the training data of that fold. This ensures that no data leakage occurs, and the evaluation remains methodologically consistent with the cross-validation strategy used throughout the modelling stage.

The confusion matrices provide insight into how each model balances the precision–recall trade-off, particularly under the moderate class imbalance identified during exploratory data analysis (Section 2.3), where adopted animals represent roughly 42% of the dataset. Models that prioritise recall produce fewer false negatives (missed adoption cases) but may increase the number of false positives, whereas models that prioritise precision tend to predict adoption more conservatively. The boosting models (XGBoost and LightGBM) clearly identify a larger proportion of adopted animals compared with the other models, producing substantially fewer false negatives. This explains their higher recall scores and reflects the use of the `scale_pos_weight` parameter, which increases the importance of correctly predicting the minority adoption class during training.

The ROC curves provide a complementary perspective by evaluating each model's ability to rank adopted animals higher than non-adopted ones across all possible classification thresholds. All models perform significantly better than the random baseline, confirming that the available predictors contain meaningful signal. Consistent with the earlier cross-validation results, XGBoost and LightGBM achieve the strongest discrimination performance, with AUC values around 0.79, followed closely by Random Forest. Logistic Regression shows the lowest AUC, indicating that a purely linear decision boundary cannot fully capture the patterns present in the data.

These findings are consistent with the patterns identified during the exploratory data analysis (Section 2.7), where several non-linear feature interactions were observed. For example, the influence of age, neuter status, intake conditions, and breed characteristics on adoption likelihood appears to depend on combinations of multiple variables rather than simple additive effects. Tree-based ensemble methods, particularly gradient boosting models, are well suited to capturing such hierarchical interactions automatically, which explains why XGBoost and LightGBM outperform the linear baseline and slightly exceed the performance of Random Forest.

Overall, the confusion matrices and ROC curves reinforce the earlier model comparison results. Gradient boosting models provide the strongest balance between precision and recall while maintaining high discriminative ability, making them the most suitable candidates for further optimisation. Consequently, the top models identified in the previous section will proceed to hyperparameter tuning in Section 5, where their configurations will be refined before final evaluation on the held-out test set.

# Fine-Tune and Evaluate

## 5.1 Hyperparameter Tuning Strategy

Based on the model comparison in Section 4, three models were shortlisted for hyperparameter tuning: XGBoost, LightGBM, and Random Forest. These models demonstrated the strongest cross-validated F1-scores and are now refined to determine whether tuning their configurations can further improve predictive performance.

Hyperparameter tuning is performed using `RandomizedSearchCV` with the same GroupKFold (k=5) cross-validation strategy used during model exploration. Each candidate configuration is evaluated on training data only, with the preprocessing pipeline refitted inside each fold to prevent data leakage. The search optimises for F1-score, consistent with the primary evaluation metric established in Section 1. Randomised search is preferred over exhaustive grid search because it explores a broader region of the hyperparameter space within a fixed computational budget, which is particularly effective when some parameters have a stronger influence on performance than others.

The number of search iterations (`n_iter`) was initially set to 50, but was reduced to 30 after observing that the Random Forest search alone required significant computation time. Although a larger number of iterations could explore more configurations, `n_iter=30` was sufficient for this project because model screening in Section 4 had already identified a narrow set of strong candidate models, and each trial was evaluated under grouped 5-fold cross-validation, making the search computationally expensive.

**Critically, the held-out test set is not used at any point during tuning.** The best model will be selected based solely on cross-validated F1-score, and the test set will only be evaluated once — in Section 5.6 — to provide an unbiased estimate of generalisation performance. This strict separation prevents test-set selection bias.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.calibration import calibration_curve
from sklearn.metrics import classification_report

print("=== Section 5 Setup ===")
print(f"Shortlisted models: Random Forest, XGBoost, LightGBM")
print(f"Tuning method: RandomizedSearchCV (n_iter=30)")
print(f"CV strategy: GroupKFold (k=5) on training data only")
print(f"Optimisation metric: F1-score")
print(f"Test set: held out until Section 5.6")

## 5.2 Random Forest Tuning

The Random Forest model demonstrated the highest ROC-AUC in Section 4, suggesting strong ranking ability. We now tune its key hyperparameters to determine whether its F1-score can be improved. The search grid targets the number of trees, tree depth, minimum samples for splitting and leaf nodes, and the number of features considered at each split — all of which control the bias-variance trade-off in ensemble tree models.

In [ ]:
# Random Forest hyperparameter tuning
rf_param_dist = {
    'model__n_estimators': randint(100, 500),
    'model__max_depth': [None, 10, 20, 30, 40],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf': randint(1, 10),
    'model__max_features': ['sqrt', 'log2', 0.3, 0.5],
}

rf_pipe = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

print("=== Random Forest Hyperparameter Tuning ===\n")
start = time.time()
rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=30, scoring='f1',
    cv=gkf, random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
rf_search.fit(X_train, y_train, groups=train_animal_ids)
rf_tune_time = time.time() - start

print(f"Best F1 (CV): {rf_search.best_score_:.4f}")
print(f"Default F1 (Section 4): {comparison_df[comparison_df['Model']=='Random Forest']['f1_mean'].values[0]:.4f}")
print(f"Improvement: {rf_search.best_score_ - comparison_df[comparison_df['Model']=='Random Forest']['f1_mean'].values[0]:+.4f}")
print(f"\nBest parameters:")
for param, val in rf_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nTuning time: {rf_tune_time:.1f}s ({30} configurations evaluated)")

Hyperparameter tuning improved the Random Forest model from an F1-score of 0.6661 in Section 4 to 0.6874 under grouped cross-validation, indicating that the default configuration was not yet optimal for this dataset. The best-performing model used 330 trees with a maximum depth of 20, `max_features=0.3`, `min_samples_split=6`, and `min_samples_leaf=1`. This configuration suggests that the model benefits from moderately deep trees that are flexible enough to capture the non-linear feature interactions identified during the exploratory analysis, while still applying some regularisation to avoid overly fragmented decision rules. In particular, the reduced `max_features` value increases diversity between trees, improving the effectiveness of bagging through less correlated splits. The improvement in F1-score therefore reflects a better bias–variance trade-off, allowing the ensemble to model complex adoption patterns more effectively without overfitting. Because tuning was carried out using `RandomizedSearchCV` with GroupKFold on the training data only, and with preprocessing refitted inside each fold via a pipeline, the result provides a leakage-free estimate of how much tuning genuinely improved generalisation. The tuned Random Forest remains especially useful as a benchmark because it offers a strong non-linear ensemble baseline against which the boosted models in later sections can be judged.

Although the tuned Random Forest improves substantially over both its default version and the logistic regression baseline, its final standing relative to XGBoost and LightGBM will show whether bagged trees are sufficient for this problem or whether sequential boosting is needed to exploit the full structure of the data.

## 5.3 XGBoost Tuning

XGBoost achieved the joint-highest F1-score in Section 4. The tuning search targets learning rate, tree depth, subsampling ratios, and L1/L2 regularisation strengths. These parameters control how aggressively the model learns from residual errors, how much of the data and feature space each tree sees, and how strongly the model is penalised for complexity — all critical for preventing overfitting while maintaining strong predictive performance.

In [ ]:
# XGBoost hyperparameter tuning
xgb_param_dist = {
    'model__n_estimators': randint(100, 500),
    'model__max_depth': randint(3, 10),
    'model__learning_rate': uniform(0.01, 0.29),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(0.5, 1.5),
}

xgb_pipe = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', xgb.XGBClassifier(
        scale_pos_weight=scale_weight, random_state=RANDOM_STATE,
        eval_metric='logloss', use_label_encoder=False, n_jobs=-1
    ))
])

print("=== XGBoost Hyperparameter Tuning ===\n")
start = time.time()
xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_param_dist, n_iter=30, scoring='f1',
    cv=gkf, random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
xgb_search.fit(X_train, y_train, groups=train_animal_ids)
xgb_tune_time = time.time() - start

print(f"Best F1 (CV): {xgb_search.best_score_:.4f}")
print(f"Default F1 (Section 4): {comparison_df[comparison_df['Model']=='XGBoost']['f1_mean'].values[0]:.4f}")
print(f"Improvement: {xgb_search.best_score_ - comparison_df[comparison_df['Model']=='XGBoost']['f1_mean'].values[0]:+.4f}")
print(f"\nBest parameters:")
for param, val in xgb_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nTuning time: {xgb_tune_time:.1f}s ({30} configurations evaluated)")

Hyperparameter tuning produced a modest improvement in XGBoost performance, increasing the cross-validated F1-score from 0.6954 in Section 4 to 0.6982, an improvement of +0.0027. Although the gain is relatively small compared with the Random Forest tuning improvement, this outcome is expected because the baseline XGBoost configuration was already performing near the upper bound observed during earlier model comparison. In Section 4, XGBoost already achieved one of the strongest F1-scores and exhibited a favourable balance between precision and recall, suggesting that its default configuration was already well aligned with the structure of the dataset. The tuned parameters mainly refine the model's learning dynamics rather than fundamentally altering its capacity. In particular, the optimal configuration uses a low learning rate (~0.04) combined with a relatively large number of trees (451), which is a common strategy in gradient boosting: each tree makes small incremental corrections to residual errors, allowing the model to gradually learn complex patterns while reducing the risk of overfitting. The selected tree depth of 9 further indicates that moderately complex feature interactions are important for predicting adoption outcomes. This aligns with findings from the exploratory analysis, where adoption likelihood appeared to depend on combinations of attributes such as intake condition, age group, breed grouping, and time-related intake variables rather than simple linear effects. The tuned model also uses feature subsampling (`colsample_bytree` ≈ 0.61) and row subsampling (`subsample` ≈ 0.80), which introduce stochasticity into the boosting process and reduce correlation between trees, improving generalisation. Additionally, the non-zero L1 and L2 regularisation terms indicate that penalising model complexity improves stability across cross-validation folds. Overall, the small performance improvement suggests that the baseline model was already operating in a strong region of the hyperparameter space, and tuning primarily fine-tuned the bias–variance balance rather than producing a dramatic change in predictive capability. Importantly, because tuning was conducted using `RandomizedSearchCV` with GroupKFold and with preprocessing fitted inside each fold through a pipeline, the resulting performance estimates remain free from data leakage and reflect genuine improvements in generalisation to unseen animals. The continued strong performance of XGBoost relative to other models further reinforces the earlier conclusion that boosting-based tree ensembles are particularly well suited to this problem, as they can effectively capture the complex, non-linear feature interactions identified during exploratory analysis while maintaining robust predictive performance.

## 5.4 LightGBM Tuning

LightGBM matched XGBoost's F1-score in Section 4 while using leaf-wise tree growth. The tuning search covers similar parameters but also includes `num_leaves`, which directly controls the complexity of individual trees in LightGBM's leaf-wise strategy. A higher number of leaves allows the model to capture finer patterns but increases the risk of overfitting, making it an important parameter to optimise alongside regularisation.

In [ ]:
# LightGBM hyperparameter tuning
lgb_param_dist = {
    'model__n_estimators': randint(100, 500),
    'model__max_depth': randint(3, 10),
    'model__learning_rate': uniform(0.01, 0.29),
    'model__num_leaves': randint(20, 80),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__reg_alpha': uniform(0, 1),
    'model__reg_lambda': uniform(0.5, 1.5),
}

lgb_pipe = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', lgb.LGBMClassifier(
        scale_pos_weight=scale_weight, random_state=RANDOM_STATE,
        verbose=-1, n_jobs=-1
    ))
])

print("=== LightGBM Hyperparameter Tuning ===\n")
start = time.time()
lgb_search = RandomizedSearchCV(
    lgb_pipe, lgb_param_dist, n_iter=30, scoring='f1',
    cv=gkf, random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
lgb_search.fit(X_train, y_train, groups=train_animal_ids)
lgb_tune_time = time.time() - start

print(f"Best F1 (CV): {lgb_search.best_score_:.4f}")
print(f"Default F1 (Section 4): {comparison_df[comparison_df['Model']=='LightGBM']['f1_mean'].values[0]:.4f}")
print(f"Improvement: {lgb_search.best_score_ - comparison_df[comparison_df['Model']=='LightGBM']['f1_mean'].values[0]:+.4f}")
print(f"\nBest parameters:")
for param, val in lgb_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nTuning time: {lgb_tune_time:.1f}s ({30} configurations evaluated)")

Hyperparameter tuning produced a modest improvement in LightGBM performance, increasing the cross-validated F1-score from 0.6952 in Section 4 to 0.6977, an improvement of +0.0025. Similar to the results observed for XGBoost tuning, the gain is relatively small, which is expected given that the baseline LightGBM configuration was already performing near the top of the model comparison in Section 4. In the earlier experiments, LightGBM matched XGBoost's performance and substantially outperformed simpler models such as logistic regression, indicating that gradient boosting methods were already well suited to the structure of the dataset. As a result, hyperparameter tuning mainly refined the model's learning dynamics and regularisation rather than producing a dramatic improvement in predictive performance. The optimal configuration combines a moderately low learning rate (~0.064) with a relatively large number of trees (430), reflecting the common boosting strategy of learning gradually through many small updates rather than relying on large corrections at each iteration. This slower learning process tends to improve generalisation by allowing the model to capture complex patterns without overfitting to noise in the training data. The selected tree depth (8) and relatively large number of leaves (77) indicate that LightGBM benefits from moderately complex tree structures capable of modelling higher-order feature interactions. This finding aligns with patterns identified during the exploratory analysis, where adoption outcomes appeared to depend on combinations of attributes such as intake condition, age group, breed grouping, and time-related intake characteristics rather than simple linear relationships between individual variables. LightGBM's leaf-wise tree growth strategy is particularly effective in capturing such interactions because it expands the most informative branches first, allowing the model to focus on the most predictive regions of the feature space. The tuned model also employs feature subsampling (`colsample_bytree` ≈ 0.92) and row subsampling (`subsample` ≈ 0.96), introducing controlled randomness that helps reduce overfitting while maintaining strong predictive signal. Additionally, the presence of non-zero L1 and L2 regularisation parameters indicates that moderate penalisation of model complexity improves stability across cross-validation folds. The relatively small improvement in F1-score therefore suggests that the baseline LightGBM configuration was already operating in a strong region of the hyperparameter space, with tuning primarily fine-tuning the bias–variance balance rather than fundamentally changing the model's behaviour. Importantly, because tuning was conducted using `RandomizedSearchCV` with GroupKFold and preprocessing refitted within each fold through a pipeline, the resulting performance estimates remain free from data leakage and provide a reliable estimate of generalisation performance. Overall, the consistently strong results for both XGBoost and LightGBM reinforce the earlier conclusion that boosting-based tree ensembles are particularly effective for this problem, as they are well suited to modelling the complex, non-linear feature interactions and heterogeneous patterns identified during the exploratory analysis while maintaining robust predictive performance under grouped cross-validation.

## 5.5 Tuned Model Comparison and Selection

With all three shortlisted models tuned, we now compare their best cross-validated F1-scores to select the final model. This selection is made entirely on training data — the test set has not been used at any point during model exploration or tuning. By choosing the best model based solely on cross-validated performance, we avoid test-set selection bias and ensure that the final evaluation in Section 5.6 provides a genuinely unbiased estimate of how the model would perform on new, unseen animals.

In [ ]:
# Compare tuned models — selection based on CV F1 only (no test data)
tuned_results = {
    'Random Forest': {'cv_f1': rf_search.best_score_, 'search': rf_search, 'time': rf_tune_time},
    'XGBoost': {'cv_f1': xgb_search.best_score_, 'search': xgb_search, 'time': xgb_tune_time},
    'LightGBM': {'cv_f1': lgb_search.best_score_, 'search': lgb_search, 'time': lgb_tune_time},
}

print("=" * 70)
print("TUNED MODEL COMPARISON — Best CV F1-Score (Training Data Only)")
print("=" * 70)
print(f"\n{'Model':<20s} {'Default F1':>12s} {'Tuned F1':>12s} {'Change':>10s} {'Time':>8s}")
print("-" * 70)

for name, res in sorted(tuned_results.items(), key=lambda x: x[1]['cv_f1'], reverse=True):
    default_f1 = comparison_df[comparison_df['Model']==name]['f1_mean'].values[0]
    change = res['cv_f1'] - default_f1
    print(f"{name:<20s} {default_f1:>12.4f} {res['cv_f1']:>12.4f} {change:>+10.4f} {res['time']:>6.1f}s")

# Select best model
best_name = max(tuned_results, key=lambda x: tuned_results[x]['cv_f1'])
best_search = tuned_results[best_name]['search']
best_pipeline = best_search.best_estimator_

print(f"\n{'=' * 70}")
print(f"SELECTED MODEL: {best_name} (CV F1 = {tuned_results[best_name]['cv_f1']:.4f})")
print(f"{'=' * 70}")
print(f"\nThis model was selected based on cross-validated F1-score alone.")
print(f"The test set has NOT been used for model selection.")

After tuning the three shortlisted models — Random Forest, XGBoost, and LightGBM — their best cross-validated F1-scores were compared to determine the final model. This comparison is based exclusively on training data using GroupKFold cross-validation, and the held-out test set has not been used at any point during model exploration, tuning, or selection. Maintaining this strict separation ensures that the final evaluation in the next section provides an unbiased estimate of how the model performs on genuinely unseen animals.

Among the tuned models, XGBoost achieved the highest cross-validated F1-score (0.6982), narrowly outperforming LightGBM (0.6977) and Random Forest (0.6874). Although the difference between XGBoost and LightGBM is small, this pattern is consistent with earlier results observed in Section 4, where both boosting models demonstrated the strongest overall performance. These models significantly outperformed simpler approaches such as logistic regression and the neural network baseline, reinforcing the earlier conclusion that tree-based ensemble methods are particularly well suited to this problem.

The superior performance of boosting models can be explained by the nature of the adoption prediction task. During exploratory data analysis, adoption outcomes were found to depend on complex, non-linear interactions between multiple predictors, including age group, intake condition, breed grouping, and time-related intake features. Unlike linear models, which assume additive relationships between predictors, boosting algorithms sequentially construct decision trees that focus on correcting the residual errors of previous trees. This iterative learning process allows boosting models to gradually capture subtle interaction effects and conditional relationships within the data.

While Random Forest also models non-linear relationships, its bagging-based ensemble strategy builds trees independently and averages their predictions. In contrast, boosting methods such as XGBoost and LightGBM build trees sequentially, allowing each new tree to focus specifically on the observations that previous trees struggled to classify correctly. This targeted error correction often results in stronger predictive performance when the underlying data structure contains complex patterns, as appears to be the case in this dataset.

Hyperparameter tuning improved all three models to some extent, though the magnitude of improvement varied. Random Forest experienced the largest increase in F1-score (+0.0213), indicating that the default configuration was not optimal for this dataset. In contrast, the improvements for XGBoost (+0.0027) and LightGBM (+0.0025) were relatively small. This suggests that the baseline configurations of the boosting models were already operating near an effective region of the hyperparameter space, and tuning primarily refined their learning rates, regularisation strengths, and tree structures rather than dramatically altering their behaviour.

Between the two boosting approaches, XGBoost ultimately achieved the highest cross-validated F1-score and is therefore selected as the final model. The difference from LightGBM is small but consistent across folds, and selecting the model with the highest validation performance provides the most objective and reproducible decision rule. Importantly, this decision is based solely on cross-validated training performance, ensuring that the held-out test set remains a completely untouched benchmark for final evaluation.

The selected XGBoost model therefore represents the configuration that best balances precision and recall for predicting adoption outcomes under the grouped cross-validation framework used throughout this analysis. In the next section, this model will be evaluated on the held-out test set to obtain an unbiased estimate of its generalisation performance on new animals.

## 5.6 Final Evaluation on Held-Out Test Set

This is the first and only time the held-out test set is used. The selected model has already been fit on the full training data during `RandomizedSearchCV` (the `best_estimator_` attribute retains the refit on the entire training set). We now evaluate it on `X_test` to obtain an unbiased estimate of generalisation performance on genuinely unseen animals — none of which appeared in training due to the `GroupShuffleSplit` in Section 3.3.

The evaluation includes a classification report (precision, recall, F1 per class), confusion matrix, ROC curve, and calibration curve. The calibration curve assesses whether the model's predicted probabilities are reliable — for example, among animals predicted to have a 70% chance of adoption, do approximately 70% actually get adopted? Good calibration is important if the predictions are used for downstream decision-making.

In [ ]:
# Final evaluation on held-out test set
print("=" * 70)
print("FINAL EVALUATION — Held-Out Test Set (First and Only Use)")
print("=" * 70)

# Predict on test set
y_test_pred = best_pipeline.predict(X_test)
y_test_proba = best_pipeline.predict_proba(X_test)[:, 1]

# Classification report
print(f"\nModel: {best_name}\n")
print(classification_report(y_test, y_test_pred, target_names=['Not Adopted', 'Adopted'], digits=4))

The selected XGBoost model was evaluated on the held-out test set to obtain an unbiased estimate of its performance on unseen animals. Importantly, this is the first and only stage in the workflow where the test data is used, ensuring that the evaluation reflects genuine generalisation performance rather than optimisation on the evaluation data. All model development, tuning, and selection decisions were based exclusively on cross-validation within the training set using GroupKFold, meaning the test results represent a realistic assessment of how the model would perform when deployed on new adoption records.

On the test set, the model achieved an overall accuracy of 0.718, with a weighted F1-score of approximately 0.719. These results are consistent with the cross-validated performance observed during model tuning, where the best XGBoost configuration achieved an F1-score of roughly 0.698 under grouped cross-validation. The slightly higher score on the test set is within the range of expected variation between folds and indicates that the model generalises well rather than overfitting the training data.

Examining the class-specific metrics provides additional insight into the model's behaviour. For the "Adopted" class, the model achieves a recall of 0.791, meaning that approximately 79% of animals that were actually adopted were correctly identified by the model. This relatively high recall is particularly important in the context of adoption prediction, as it indicates that the model is effective at identifying animals with a high likelihood of being adopted. However, this improvement in recall comes with a precision of 0.634, reflecting a trade-off in which some animals predicted to be adopted were not ultimately adopted. This trade-off between precision and recall is expected when models are optimised for F1-score, which balances both metrics rather than prioritising one over the other.

For the "Not Adopted" class, the model achieves a precision of 0.812 and recall of 0.664, indicating that predictions of non-adoption are generally reliable but that some animals that were not adopted are incorrectly predicted as adoptable. This asymmetry between classes reflects the model's tendency to favour identifying potential adoptions, which is consistent with the optimisation objective used during training and with the moderate class imbalance observed in the dataset during exploratory analysis.

Overall, the model demonstrates a balanced performance across both classes, with F1-scores of 0.731 for the "Not Adopted" class and 0.703 for the "Adopted" class. These results confirm that the XGBoost model is able to capture meaningful patterns in the data and generalise effectively to new observations. The strong performance of the boosting model also reinforces earlier findings from the exploratory analysis and model comparison stages, where complex interactions between features such as intake condition, age group, breed grouping, and temporal intake characteristics appeared to influence adoption outcomes. Gradient boosting methods are particularly well suited to modelling such non-linear interactions, which likely explains their superior performance relative to simpler models such as logistic regression.

Importantly, because the evaluation was conducted on a fully held-out test set that was never used during training or tuning, these results provide a credible estimate of how the model would perform in a real-world deployment scenario. In the next section, the model's predictions will be analysed in more detail through confusion matrices, ROC curves, calibration analysis, and error analysis to better understand its strengths, limitations, and typical failure modes.

In [ ]:
# Test set visualisations: Confusion Matrix, ROC Curve, Calibration Curve
from sklearn.metrics import precision_score, recall_score, f1_score as f1_metric, roc_auc_score as auc_metric

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Adopted', 'Adopted'])
disp.plot(ax=axes[0], cmap='Blues', values_format=',d', colorbar=False)
axes[0].set_title(f'Confusion Matrix — {best_name}\n(Test Set)', fontweight='bold')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
roc_auc_test = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'{best_name} (AUC = {roc_auc_test:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Test Set', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

# 3. Calibration Curve
prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=10, strategy='uniform')
axes[2].plot(prob_pred, prob_true, 's-', color='#3498db', lw=2, label=best_name)
axes[2].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Perfectly calibrated')
axes[2].set_xlabel('Mean Predicted Probability')
axes[2].set_ylabel('Fraction of Positives')
axes[2].set_title('Calibration Curve — Test Set', fontweight='bold')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key metrics summary
test_f1 = f1_metric(y_test, y_test_pred)
test_auc = auc_metric(y_test, y_test_proba)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)

print(f"\n=== Test Set Summary ===")
print(f"  F1-Score:  {test_f1:.4f}")
print(f"  AUC-ROC:   {test_auc:.4f}")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"\n  CV F1 (training): {tuned_results[best_name]['cv_f1']:.4f}")
print(f"  Test F1:          {test_f1:.4f}")
print(f"  Difference:       {test_f1 - tuned_results[best_name]['cv_f1']:+.4f}")
print(f"\n  A small difference between CV and test F1 indicates the model")
print(f"  generalises well and is not overfitting to the training data.")

The tuned XGBoost model demonstrates strong and well-balanced performance on the held-out test set. The model achieves an F1-score of 0.703, an AUC-ROC of 0.797, precision of 0.634, and recall of 0.791 for the adopted class. These results are highly consistent with the cross-validated training performance (CV F1 = 0.698), with only a small difference of +0.0053 between CV and test F1. This small gap suggests that the model generalises well to unseen animals and is not overfitting the training data. The stability between cross-validation and test performance further confirms that the GroupKFold validation strategy successfully prevented information leakage between observations belonging to the same animal.

The confusion matrix provides additional insight into the model's classification behaviour. Of the animals that were actually adopted, 5,353 were correctly predicted as adopted, while 1,417 were incorrectly predicted as not adopted, representing the model's false negatives. This corresponds to a recall of approximately 79% for the adopted class, indicating that the model is effective at identifying animals that are likely to be adopted. However, the model also produces 3,097 false positives, where animals predicted as adoptable were not actually adopted. This reflects the precision–recall trade-off inherent in optimising for F1-score: increasing recall improves the model's ability to detect adoptable animals but also increases the number of incorrect adoption predictions. In the context of adoption prediction, prioritising recall can be desirable because failing to identify animals with high adoption potential may represent a more costly error than incorrectly flagging some animals as adoptable.

The ROC curve further demonstrates the model's ability to distinguish between adopted and non-adopted outcomes across different classification thresholds. The AUC-ROC of 0.797 indicates strong discriminative performance, meaning the model can effectively rank animals by their likelihood of adoption. In practical terms, this suggests that if the model assigns higher predicted probabilities to animals that are ultimately adopted, decision-makers could use these scores to prioritise animals for promotion or targeted interventions.

The calibration curve assesses whether the predicted probabilities correspond to actual adoption frequencies. The curve follows the diagonal reasonably closely, indicating that the predicted probabilities are broadly well calibrated. This means that when the model predicts an adoption probability of, for example, 70%, roughly 70% of animals in that probability range are actually adopted. Good calibration is important when predicted probabilities are used for decision-making rather than just classification, as it allows shelters to interpret model outputs as meaningful estimates of adoption likelihood.

Overall, the test set evaluation confirms that the selected XGBoost model generalises well and captures meaningful predictive patterns in the data. The model's strong recall for adopted animals and good discriminative ability are consistent with earlier findings from the exploratory analysis and model comparison stages, where adoption outcomes appeared to depend on complex, non-linear interactions between features such as intake condition, age group, breed grouping, and temporal intake characteristics. Gradient boosting models are particularly well suited to capturing such interactions, which likely explains their superior performance relative to simpler models tested earlier in the analysis.

## 5.7 Error Analysis

To understand where the model succeeds and fails, we analyse its errors on the test set. Specifically, we examine false negatives (animals that were adopted but the model predicted would not be) and false positives (animals predicted to be adopted but were not). Identifying systematic patterns in these misclassifications can reveal limitations of the model and suggest directions for future improvement. We also extract feature importances from the best model to understand which predictors most strongly influence adoption predictions.

In [ ]:
# Error analysis: False Negatives and False Positives
test_results = X_test.copy()
test_results['actual'] = y_test.values
test_results['predicted'] = y_test_pred
test_results['prob_adopted'] = y_test_proba

# Classify prediction outcomes
test_results['outcome'] = 'True Negative'
test_results.loc[(test_results['actual'] == 1) & (test_results['predicted'] == 1), 'outcome'] = 'True Positive'
test_results.loc[(test_results['actual'] == 1) & (test_results['predicted'] == 0), 'outcome'] = 'False Negative'
test_results.loc[(test_results['actual'] == 0) & (test_results['predicted'] == 1), 'outcome'] = 'False Positive'

print("=== Prediction Outcome Distribution (Test Set) ===\n")
print(test_results['outcome'].value_counts().to_string())

# Analyse False Negatives — missed adoptions
fn = test_results[test_results['outcome'] == 'False Negative']
fp = test_results[test_results['outcome'] == 'False Positive']

print(f"\n=== False Negatives (Missed Adoptions): {len(fn):,} cases ===")
print(f"These animals WERE adopted but the model predicted they would NOT be.\n")

for col in ['animal_type', 'is_neutered', 'intake_type', 'intake_condition']:
    if col in fn.columns:
        print(f"  {col}:")
        dist = fn[col].value_counts(normalize=True).head(5)
        for val, pct in dist.items():
            print(f"    {val}: {pct:.1%}")
        print()

print(f"  Median age (days): {fn['age_upon_intake_(days)'].median():.0f}")
print(f"  Mean predicted probability: {fn['prob_adopted'].mean():.3f}")

print(f"\n=== False Positives (Incorrect Adoption Predictions): {len(fp):,} cases ===")
print(f"These animals were NOT adopted but the model predicted they WOULD be.\n")

for col in ['animal_type', 'is_neutered', 'intake_type', 'intake_condition']:
    if col in fp.columns:
        print(f"  {col}:")
        dist = fp[col].value_counts(normalize=True).head(5)
        for val, pct in dist.items():
            print(f"    {val}: {pct:.1%}")
        print()

print(f"  Median age (days): {fp['age_upon_intake_(days)'].median():.0f}")
print(f"  Mean predicted probability: {fp['prob_adopted'].mean():.3f}")

To better understand the limitations of the final model, we analysed misclassifications on the held-out test set. In particular, we examined false negatives (animals that were adopted but predicted as not adopted) and false positives (animals predicted to be adopted but that were not). Identifying systematic patterns in these errors helps reveal where the model struggles and provides insight into the underlying structure of the prediction problem.

The confusion matrix shows 1,417 false negatives and 3,097 false positives. False negatives represent missed adoption opportunities — animals that were successfully adopted but the model predicted would not be. Analysis of these cases reveals several consistent patterns. Approximately 61% of false negatives are dogs and 37% are cats, reflecting the fact that these animals dominate the dataset overall, but also suggesting that adoption behaviour within these groups may be more difficult to predict due to variability in factors such as breed, behaviour, or presentation in the shelter. The majority of false negatives also originate from stray intake cases (≈86%), which is consistent with patterns observed during the exploratory analysis where stray animals showed more uncertain adoption outcomes due to limited background information.

Additionally, the median age of false negatives is approximately 365 days, suggesting that the model may struggle to identify adoption potential for animals around the young-adult stage. This may reflect subtle interactions between age and other features (such as neuter status or intake condition) that are difficult to fully capture even with tree-based models. Interestingly, the mean predicted adoption probability for these false negatives is relatively high (≈0.665), indicating that the model often considered these animals somewhat likely to be adopted but still classified them below the decision threshold. This suggests that some of these errors occur near the classification boundary, where small probability shifts could change the predicted class.

False positives represent the opposite error: animals predicted as likely to be adopted but that ultimately were not. These errors are partially a consequence of optimising for F1-score, which balances precision and recall. Earlier sections highlighted that the dataset exhibits moderate class imbalance, with adopted animals representing roughly 42% of observations. Without adjustments, models tend to favour predicting the majority class (non-adoption). To counter this, class weighting was applied during training, encouraging the model to identify more adoption cases. As a result, the model achieves strong recall for the adopted class but accepts a higher number of false positives — a deliberate precision–recall trade-off that prioritises detecting animals with adoption potential.

These error patterns also reinforce findings from the exploratory data analysis. Earlier analysis showed that adoption outcomes depend on complex interactions between variables such as intake type, age, neuter status, and intake condition, rather than any single predictor. While gradient boosting models like XGBoost are effective at capturing such non-linear relationships, the variability in real-world adoption outcomes means that some cases remain inherently difficult to predict. Factors not captured in the dataset — such as behavioural traits, shelter marketing, or adopter preferences — may also contribute to these remaining errors.

Overall, the error analysis suggests that the model performs reliably but still faces challenges in borderline cases where adoption outcomes depend on nuanced or unobserved factors. Future improvements could include incorporating additional behavioural or descriptive features, adjusting classification thresholds depending on operational priorities, or using probability-based ranking rather than strict binary classification to prioritise animals with the highest adoption likelihood.

These findings highlight that adoption prediction is inherently probabilistic, and even well-performing models may struggle with cases where outcomes depend on human decision-making or contextual factors not captured in structured shelter data.

In [ ]:
# Feature importance from best model
best_model = best_pipeline.named_steps['model']

# Get feature names from the preprocessor
ohe_step = best_pipeline.named_steps['preprocess'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = ohe_step.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + binary_features + cat_feature_names

# Extract importances
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    # Plot top 20
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = feat_imp.head(20)
    ax.barh(range(len(top_n)), top_n['Importance'].values, color='#3498db', edgecolor='white')
    ax.set_yticks(range(len(top_n)))
    ax.set_yticklabels(top_n['Feature'].values, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 20 Feature Importances — {best_name}', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"\n=== Top 10 Features ===\n")
    for i, (_, row) in enumerate(feat_imp.head(10).iterrows(), 1):
        print(f"  {i:2d}. {row['Feature']:<40s} {row['Importance']:.4f}")
else:
    print("Feature importances not available for this model type.")

In [ ]:
# Error analysis visualisation: FN vs TP by key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tp = test_results[test_results['outcome'] == 'True Positive']

# 1. Age distribution: FN vs TP
axes[0, 0].hist(tp['age_upon_intake_(days)'], bins=50, alpha=0.6, label='True Positive', color='#2ecc71', density=True)
axes[0, 0].hist(fn['age_upon_intake_(days)'], bins=50, alpha=0.6, label='False Negative', color='#e74c3c', density=True)
axes[0, 0].set_xlabel('Age (days)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Age Distribution: Correct vs Missed Adoptions', fontweight='bold')
axes[0, 0].legend()

# 2. Animal type: FN rate
animal_fn_rate = test_results[test_results['actual'] == 1].groupby('animal_type')['outcome'].apply(
    lambda x: (x == 'False Negative').mean()
).sort_values(ascending=False)
axes[0, 1].bar(range(len(animal_fn_rate)), animal_fn_rate.values, color='#e74c3c')
axes[0, 1].set_xticks(range(len(animal_fn_rate)))
axes[0, 1].set_xticklabels(animal_fn_rate.index, rotation=45, ha='right')
axes[0, 1].set_ylabel('False Negative Rate')
axes[0, 1].set_title('Missed Adoption Rate by Animal Type', fontweight='bold')

# 3. Predicted probability distribution by outcome
for outcome, color, label in [('True Positive', '#2ecc71', 'TP'),
                                ('False Negative', '#e74c3c', 'FN'),
                                ('False Positive', '#f39c12', 'FP'),
                                ('True Negative', '#3498db', 'TN')]:
    subset = test_results[test_results['outcome'] == outcome]
    if len(subset) > 0:
        axes[1, 0].hist(subset['prob_adopted'], bins=30, alpha=0.5, label=f'{label} (n={len(subset):,})',
                        density=True)
axes[1, 0].set_xlabel('Predicted Adoption Probability')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Prediction Confidence by Outcome', fontweight='bold')
axes[1, 0].legend(fontsize=8)

# 4. Intake condition: FN rate
cond_fn_rate = test_results[test_results['actual'] == 1].groupby('intake_condition')['outcome'].apply(
    lambda x: (x == 'False Negative').mean()
).sort_values(ascending=False)
axes[1, 1].bar(range(len(cond_fn_rate)), cond_fn_rate.values, color='#e74c3c')
axes[1, 1].set_xticks(range(len(cond_fn_rate)))
axes[1, 1].set_xticklabels(cond_fn_rate.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('False Negative Rate')
axes[1, 1].set_title('Missed Adoption Rate by Intake Condition', fontweight='bold')

plt.suptitle(f'Error Analysis — {best_name} (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Feature importance extracted from the final XGBoost model provides insight into which predictors most strongly influence adoption predictions. The two most influential variables are is_neutered_Unknown (importance ≈ 0.37) and gender_Unknown (importance ≈ 0.28). This suggests that missing information about an animal's neuter status or gender plays a substantial role in the model's decision-making process. In practice, these indicators likely act as proxies for intake circumstances: animals arriving without recorded demographic information are often strays or emergency intakes, where the uncertainty surrounding background information may influence adoption outcomes. This observation aligns with patterns identified during the exploratory data analysis, where intake context and animal history were found to be strongly associated with adoption likelihood.

Several intake-related features also appear among the most important predictors, including intake_type_Wildlife, intake_type_Public Assist, and intake_type_Owner Surrender. These variables capture the circumstances under which animals enter the shelter system and therefore reflect differences in adoption probability across intake pathways. For example, animals surrendered by owners may have different adoption prospects compared with stray animals, while wildlife cases often represent fundamentally different outcomes. The presence of these variables among the top predictors supports earlier EDA findings that intake type and intake condition are key contextual signals influencing adoption outcomes.

The feature importance ranking also highlights biological and physical attributes such as animal type, breed grouping, age at intake, and intake condition, although these features individually contribute less than the intake-related variables. This distribution of importance suggests that adoption outcomes are not driven by a single dominant factor but instead emerge from complex interactions between demographic, behavioural, and contextual variables. Gradient boosting models such as XGBoost are particularly well suited to capturing these non-linear relationships, which helps explain why boosting-based models performed best during model comparison.

The age distribution plot comparing true positives (correctly predicted adoptions) and false negatives (missed adoptions) provides further insight into where the model struggles. The distributions show that missed adoption cases occur across a wide range of ages, although there is some concentration among animals in the young to middle-age range. This pattern suggests that while very young animals may have relatively predictable adoption outcomes, animals in intermediate age ranges may exhibit more variability in adoption likelihood depending on other factors such as breed, condition, or behavioural characteristics.

The missed adoption rate by animal type plot indicates that false negative rates are highest for birds and "other" animal categories, while dogs and cats exhibit lower error rates. This likely reflects the much smaller sample sizes and greater heterogeneity within these less common animal categories, which makes it more difficult for the model to learn stable predictive patterns. Dogs and cats, which dominate the dataset, provide more training examples and therefore allow the model to capture clearer adoption signals.

The prediction confidence distribution further illustrates how the model behaves near the classification threshold. True positive predictions generally cluster at higher predicted probabilities, while true negatives cluster near lower probabilities, indicating that the model is able to separate many cases effectively. However, false positives and false negatives tend to occur near the middle probability range, suggesting that these misclassifications arise primarily in borderline cases where the model has moderate uncertainty about the outcome. This behaviour is expected in real-world classification tasks where some observations cannot be cleanly separated based on available features.

Finally, the missed adoption rate by intake condition reveals that animals entering the shelter with conditions labelled as "Other," "Aged," "Sick," or "Injured" exhibit substantially higher false negative rates than animals recorded as "Normal." This suggests that the model may systematically underestimate adoption likelihood for animals with health-related or complex intake conditions. One possible explanation is that these animals have more variable adoption outcomes, depending on factors not captured in the dataset, such as veterinary treatment success, behavioural rehabilitation, or targeted promotion efforts by shelter staff.

Taken together, these findings highlight that while the model performs well overall, prediction errors tend to arise in cases with incomplete information, rare animal categories, or complex health conditions. Many of these situations involve factors that may not be fully represented in structured shelter data. Future improvements could therefore involve incorporating additional behavioural or medical indicators, improving the completeness of intake records, or modelling temporal adoption dynamics. Despite these limitations, the analysis confirms that the model captures meaningful adoption patterns while maintaining strong generalisation performance on unseen data.

## 5.8 Section 5 Summary

This section focused on refining and evaluating the strongest models identified during earlier experiments. Hyperparameter tuning was conducted to optimise predictive performance while maintaining a strict separation between training and test data to avoid selection bias. The final evaluation confirmed that gradient boosting models were best suited to the dataset, capturing the complex feature interactions identified during exploratory analysis.

**Hyperparameter tuning:**
Three models — Random Forest, XGBoost, and LightGBM — were tuned using RandomizedSearchCV with GroupKFold cross-validation (k = 5) on the training data only. The search optimised F1-score, which was selected as the primary metric due to the moderate class imbalance in adoption outcomes. Randomised search allowed efficient exploration of the hyperparameter space while keeping computational cost manageable. The best model was selected exclusively based on cross-validated F1 performance, ensuring the test set remained completely unseen during model development.

**Final evaluation:**
The selected model was evaluated once on the held-out test set, providing an unbiased estimate of real-world performance. Evaluation metrics included the classification report, confusion matrix, ROC curve, and calibration curve. The final XGBoost model achieved strong performance with balanced precision and recall for the adopted class. Importantly, the small difference between cross-validation F1 and test F1 indicates good generalisation, suggesting the model is not overfitting and can reliably predict adoption outcomes for unseen animals.

**Error analysis:**
To better understand model behaviour, prediction errors were analysed in detail. False negatives (missed adoptions) and false positives were examined across key variables including animal type, age, intake condition, and neuter status. Feature importance analysis revealed that intake context and missing demographic information (e.g., unknown neuter status or gender) were among the strongest predictors of adoption outcomes. Additional visualisations of prediction confidence and error distributions showed that most misclassifications occur in borderline cases with moderate predicted probabilities, reflecting inherent uncertainty in real-world adoption decisions. These findings suggest that while the model captures meaningful patterns, adoption outcomes are also influenced by external factors not fully represented in structured shelter data.

Overall, the results demonstrate that boosting-based models effectively capture the non-linear relationships and feature interactions present in the dataset, leading to strong predictive performance and reliable generalisation on unseen data. These results highlight the potential for machine learning models to support shelter decision-making by identifying animals with high adoption potential, while also emphasising the importance of combining predictive insights with human judgement in complex real-world environments.

# Section 6: Present the Final Solution

## 6.1 Problem Recap and Solution Overview

This project addressed the problem of predicting whether an animal entering the Austin Animal Center shelter system will ultimately be adopted. Accurately predicting adoption outcomes has practical value for shelter operations: it can help staff prioritise resources, identify animals that may require additional support or promotion to find homes, and ultimately improve welfare outcomes for animals in care. By anticipating which animals are most likely to be adopted, shelters can make more informed decisions about intake management, fostering programmes, and outreach efforts.

To address this task, a predictive solution was developed through a structured end-to-end workflow:

1. **Data collection and problem framing** — The Austin Animal Center Intakes and Outcomes dataset was obtained and the prediction task was formulated as a binary classification problem (Adopted vs Not Adopted).
2. **Exploratory data analysis** — The dataset was examined to understand feature distributions, relationships between variables, and key patterns associated with adoption outcomes.
3. **Feature engineering and preprocessing** — Raw variables were transformed into meaningful predictive features, high-cardinality categories were grouped, and a preprocessing pipeline was constructed to handle both numerical and categorical inputs.
4. **Train–test split** — Data was partitioned using GroupShuffleSplit to prevent data leakage from repeated animals, ensuring that all subsequent modelling steps were performed exclusively on training data.
5. **Model exploration** — Five candidate models were evaluated using GroupKFold cross-validation, and the strongest performers were shortlisted based on F1-score.
6. **Hyperparameter tuning and evaluation** — The shortlisted models were tuned using randomised hyperparameter search. The best model was selected based on cross-validated performance, and a single final evaluation was conducted on the held-out test set to estimate generalisation performance.

The following subsections present the final selected model, summarise the key insights derived from the analysis, discuss limitations of the approach, and outline potential directions for future improvements.

## 6.2 Final Model Summary

The best-performing model across all stages of experimentation was **XGBoost** (Extreme Gradient Boosting), selected based on its cross-validated F1-score during hyperparameter tuning. The table below summarises the tuned hyperparameters and the key performance metrics on both cross-validation and the held-out test set.

**Tuned Hyperparameters:**

| Hyperparameter | Description | Tuned Value |
|:---|:---|:---|
| `n_estimators` | Number of boosting rounds | Tuned via search (100-500) |
| `max_depth` | Maximum tree depth | Tuned via search (3-10) |
| `learning_rate` | Step size shrinkage | Tuned via search (0.01-0.30) |
| `subsample` | Row subsampling ratio | Tuned via search (0.6-1.0) |
| `colsample_bytree` | Feature subsampling ratio | Tuned via search (0.6-1.0) |
| `reg_alpha` | L1 regularisation | Tuned via search (0-1) |
| `reg_lambda` | L2 regularisation | Tuned via search (0.5-2.0) |
| `scale_pos_weight` | Class imbalance adjustment | ~1.38 (ratio of majority to minority class) |

**Performance Metrics:**

| Metric | Value |
|:---|:---|
| Cross-Validated F1 (Training) | 0.6982 |
| Test Accuracy | 0.718 |
| Test F1 (Adopted) | 0.703 |
| Test ROC-AUC | 0.797 |
| CV-Test F1 Gap | -0.005 |

The best-performing model across all stages of experimentation was XGBoost (Extreme Gradient Boosting), which was selected based on its superior cross-validated F1-score during hyperparameter tuning. XGBoost is a gradient boosting algorithm that constructs an ensemble of decision trees sequentially, where each new tree is trained to correct the residual errors of the previous trees. This iterative learning process allows the model to capture complex nonlinear relationships between features while maintaining strong predictive performance.

During the tuning process, several key hyperparameters were optimised using randomised search to balance model complexity and generalisation ability. Parameters such as `max_depth` and `n_estimators` control the structure and size of the tree ensemble, while `learning_rate` regulates the contribution of each boosting step to prevent overly aggressive fitting. Subsampling parameters (`subsample` and `colsample_bytree`) were also tuned to introduce stochasticity into the training process, which helps reduce overfitting and improves robustness. Additionally, L1 and L2 regularisation terms (`reg_alpha` and `reg_lambda`) were included to penalise overly complex trees. Because the dataset contains a modest class imbalance between adopted and non-adopted animals, the `scale_pos_weight` parameter was used to slightly upweight the minority class during training, encouraging the model to better identify adoptable animals.

The final tuned model achieved a cross-validated F1-score of 0.6982 on the training data. When evaluated on the held-out test set, which was not used during model development or selection, the model achieved a test F1-score of 0.703, ROC-AUC of 0.797, and overall accuracy of 71.8%. These results indicate that the model is capable of distinguishing between animals likely to be adopted and those that are not with reasonably strong predictive performance.

Importantly, the difference between the cross-validated F1-score and the test F1-score is very small (approximately 0.005). This minimal gap suggests that the model generalises well to unseen data and is not overfitting the training set. In practical terms, the model's performance on the test set closely matches what was observed during cross-validation, indicating that the evaluation procedure and modelling pipeline produced a reliable estimate of real-world performance.

Looking more closely at the class-level metrics, precision and recall for the adopted class are relatively well balanced, meaning that the model can successfully identify many animals that will ultimately be adopted while avoiding an excessive number of false positive predictions. This balance is particularly important for shelter decision-making: a model that predicts adoption too aggressively could misallocate resources, whereas one that is too conservative could fail to highlight animals that would benefit from additional promotion or support.

Overall, these results demonstrate that the tuned XGBoost model provides a robust predictive solution for estimating adoption outcomes, offering both strong classification performance and good generalisation to previously unseen animals.

## 6.3 Key Findings and Insights

The analysis revealed several important insights into the factors influencing adoption outcomes at the Austin Animal Center.

**1. Intake context is the strongest predictor of adoption outcomes.** Feature importance analysis showed that variables related to intake circumstances — particularly unknown neuter status (`is_neutered_Unknown`) and unknown gender (`gender_Unknown`) — were by far the most influential predictors. These variables likely act as proxies for the conditions under which animals enter the shelter system. For example, stray or emergency intakes often lack complete demographic information, and this uncertainty appears strongly associated with lower adoption likelihood. Intake pathway variables (such as owner surrender, stray, or wildlife intake) also ranked highly, reinforcing the idea that the circumstances of entry into the shelter system play a critical role in shaping eventual outcomes.

**2. Adoption outcomes arise from complex feature interactions rather than single predictors.** No single variable determines adoption outcomes in isolation. Instead, outcomes emerge from interactions between demographic attributes (such as age, breed, and animal type), health-related intake conditions (e.g., normal, sick, injured), and contextual factors such as intake type and neuter status. This finding helps explain why tree-based ensemble models — particularly gradient boosting methods like XGBoost — consistently outperformed simpler linear models. These models are specifically designed to capture non-linear relationships and interaction effects that are common in real-world decision processes such as animal adoption.

**3. Class imbalance and probability thresholds influence model behaviour.** With approximately 42% of outcomes resulting in adoption, the dataset exhibits moderate class imbalance. Addressing this imbalance through the `scale_pos_weight` parameter and selecting F1-score as the primary evaluation metric helped the model balance precision and recall for the adopted class. Many prediction errors occur near the classification threshold, where the model assigns moderate probabilities of adoption. This suggests that a substantial portion of cases lie near the decision boundary and may be inherently difficult to classify based on the available features.

**4. Certain animal subgroups are systematically harder to predict.** Error analysis revealed higher misclassification rates among animals with health-related intake conditions (such as sick, injured, or aged animals), as well as among rare animal categories (e.g., birds or "other" species). These groups likely exhibit greater variability in adoption outcomes, driven by factors not captured in the structured shelter dataset. For example, behavioural traits, veterinary treatment success, individual adopter preferences, and targeted promotional efforts may substantially influence adoption likelihood but are not represented in the available features.

**5. Exploratory findings were validated by model behaviour.** Patterns identified during exploratory data analysis — including the strong relationship between intake type and adoption outcomes, the influence of neuter status, and the variability in outcomes for stray animals — were consistently reflected in both the model's feature importance rankings and the observed error patterns. This alignment between exploratory insights and model behaviour strengthens confidence in the robustness of the analytical approach and suggests that the model is capturing meaningful underlying patterns in the data rather than spurious correlations.

## 6.4 Limitations

While the final model demonstrates strong predictive performance, several limitations of the dataset, methodology, and modelling approach should be acknowledged.

**Data limitations:**
- **Limited feature scope.** The dataset consists primarily of structured administrative records. Important factors that likely influence adoption outcomes — such as animal temperament, behavioural assessments, physical appearance, and shelter marketing or promotion efforts — are not represented in the available features. These unobserved variables may account for a substantial portion of prediction errors, particularly in borderline cases where adoption decisions depend on qualitative characteristics not captured in the data.
- **Loss of granularity in high-cardinality features.** High-cardinality variables such as breed, colour, and found location were grouped into the most frequent categories to reduce dimensionality and mitigate overfitting. While this approach improves model stability, it inevitably removes some detailed information. Rare breeds or locations that may carry meaningful signals for adoption outcomes are therefore partially obscured.
- **Single-institution dataset.** The dataset originates from a single shelter system (Austin Animal Center). Adoption dynamics may vary across regions due to differences in local demographics, shelter policies, community engagement, and capacity constraints. As a result, the generalisability of the model to other shelter environments has not been established.

**Methodological limitations:**
- **Limited hyperparameter search space.** Hyperparameter tuning was performed using randomised search with a limited number of iterations per model. While this approach provides a computationally efficient exploration of the parameter space, it does not guarantee identification of the globally optimal configuration. More advanced optimisation strategies — such as Bayesian optimisation or larger search budgets — could potentially yield marginal performance improvements.
- **Fixed decision threshold.** The model uses a standard classification threshold of 0.5. However, the optimal decision threshold may depend on operational priorities. For example, shelters might prioritise higher recall to identify as many potentially adoptable animals as possible, or higher precision to avoid allocating resources to animals unlikely to be adopted. Threshold optimisation was not explored but could enhance practical usefulness.
- **Manually designed feature engineering.** Feature engineering was performed manually based on domain knowledge and exploratory analysis. While effective, more systematic feature discovery techniques — such as automated feature selection, interaction modelling, or representation learning — could reveal additional predictive signals.

**Modelling limitations:**
- **Binary outcome simplification.** The model treats adoption as a static binary outcome. In reality, adoption is a time-dependent process: animals may be adopted quickly, slowly, or not at all. Methods such as survival analysis or time-to-event modelling could provide a richer understanding of adoption dynamics.
- **Absence of temporal context.** Temporal factors such as seasonal adoption patterns, day-of-week effects, and broader societal trends were not incorporated into the model. If adoption behaviour varies over time, the current model may miss important contextual signals.
- **Potential concept drift.** Adoption patterns may change over time due to evolving shelter policies, economic conditions, or public behaviour. Because the model was trained on historical data, it may require periodic retraining to maintain predictive accuracy if underlying patterns shift.

## 6.5 Future Work and Recommendations

Several directions could extend and improve the predictive solution developed in this project, both from a data perspective and in terms of modelling and practical deployment.

**Feature enrichment:**
- **Behavioural and medical data.** Incorporating structured behavioural assessments (e.g., temperament scores, sociability evaluations) and more detailed medical information (e.g., vaccination status, treatment history, recovery outcomes) could capture important signals that are currently absent from the dataset. Animals with positive behavioural profiles or completed medical treatments may exhibit substantially different adoption trajectories, and including these variables could help reduce prediction errors for borderline cases.
- **Text-based features.** Shelter records often include free-text descriptions written by staff or volunteers. Natural language processing (NLP) techniques could be applied to extract useful signals from these descriptions, such as sentiment, behavioural keywords, or descriptive attributes. These text-derived features may capture qualitative aspects of animals that are not reflected in structured variables.
- **Temporal features.** Incorporating time-related variables such as day of week, month, season, or time since intake could allow the model to capture cyclical adoption patterns. For example, adoption rates may increase during weekends, holidays, or community awareness campaigns, and accounting for these temporal effects could improve predictive performance.

**Modelling improvements:**
- **Threshold optimisation.** Rather than relying on the default classification threshold of 0.5, the decision boundary could be adjusted based on shelter-specific priorities. For instance, lowering the threshold would increase recall and help identify a larger number of potentially adoptable animals, while raising the threshold would prioritise precision and reduce false positives. The optimal threshold could be determined by analysing precision–recall trade-offs.
- **Probability-based ranking.** Instead of producing only binary predictions, the model's predicted probabilities could be used to rank animals by estimated adoption likelihood. This approach would allow shelters to prioritise resources along a continuum rather than relying on a strict yes/no classification and may be more robust in situations where cases fall close to the decision boundary.
- **Advanced hyperparameter optimisation.** Bayesian optimisation methods (e.g., Optuna or Hyperopt) could be used to explore the hyperparameter space more efficiently than randomised search, potentially identifying improved configurations within a similar computational budget.
- **Ensemble approaches.** Combining predictions from multiple tuned models (e.g., stacking or blending XGBoost, LightGBM, and Random Forest) may further improve predictive robustness by capturing complementary patterns learned by different algorithms.

**Operational recommendations:**
- **Periodic model retraining.** Adoption patterns may evolve over time due to changes in shelter policies, community behaviour, or external factors. Periodic retraining with newly collected data would help ensure that the model remains aligned with current conditions and maintains predictive accuracy.
- **Multi-shelter validation.** Evaluating the model on data from additional shelter systems would help determine whether the learned patterns generalise beyond Austin Animal Center or whether locally trained models are required for different regions.
- **Integration with shelter workflows.** In practice, the model could be deployed as a decision-support tool that estimates adoption likelihood at intake. Such predictions could help staff prioritise animals for fostering, targeted promotion, or behavioural enrichment programmes. Importantly, these predictions should complement — rather than replace — human expertise, particularly in complex cases where contextual judgement is required.

## 6.6 Conclusion

This project developed an end-to-end predictive analytics framework for estimating animal adoption outcomes at the Austin Animal Center. Beginning with raw shelter intake and outcome records, the analysis progressed through structured exploratory data analysis, targeted feature engineering, rigorous model comparison, and systematic hyperparameter tuning, ultimately resulting in a tuned XGBoost model.

The final model achieved a test F1-score of 0.703 and a ROC-AUC of 0.797, with only a minimal difference between cross-validated and test performance. This close alignment indicates that the model generalises well to previously unseen data and is not overfitting the training set. Feature importance analysis showed that intake context — particularly the completeness of demographic information and the pathway through which animals enter the shelter system — plays a dominant role in predicting adoption outcomes. Error analysis further revealed that misclassifications tend to occur in borderline cases involving animals with incomplete records, health-related intake conditions, or less common animal categories.

Methodological rigour was maintained throughout the modelling process by enforcing strict separation between training and test data, applying group-aware cross-validation to prevent leakage from repeated animals, and selecting the final model solely on cross-validated performance. These safeguards ensure that the reported evaluation metrics provide a realistic estimate of how the model would perform when applied to new shelter intake cases.

At the same time, the analysis highlights several limitations. Important factors influencing adoption — such as behavioural traits, visual appearance, and targeted promotional efforts — are not captured in the available data, and the dataset represents only a single shelter system. These constraints suggest that future work incorporating richer behavioural data, temporal features, and cross-shelter validation could further strengthen predictive performance and improve generalisability.

Overall, the results demonstrate that machine learning methods can uncover meaningful patterns in shelter data and provide actionable insights for animal welfare organisations. By identifying animals with higher or lower predicted adoption likelihood at the point of intake, such models could support more informed resource allocation, targeted intervention strategies, and ultimately improved welfare outcomes for animals in shelter care.